# Modelo de Fellegi–Sunter para vincular registros de personas

Este notebook es la especificación del modelo implementado en `src/zona4_graph_loader/fellegi_sunter/`. Define qué se compara en cada par de registros, el modelo probabilístico, cómo se estiman sus parámetros y cómo se decide. El código implementa lo que está escrito acá.

Las celdas de código con la etiqueta `contrato` atan el texto al código: afirman, con ejemplos chicos calculados a mano, lo que el código tiene que hacer. `tests/fellegi_sunter/test_contrato_notebook.py` las ejecuta en orden, sin Jupyter, y falla si dejan de cumplirse.

## 0. Qué hace y qué no hace

Esta sección dice el alcance del modelo y cómo usar el notebook para auditarlo.

**Qué hace.** Para cada par de registros de personas que vienen de fuentes distintas, tienen roles compatibles y pasan un filtro por apellido, estima la probabilidad de que describan a la misma persona y propone una de tres decisiones: vincular, revisar o no vincular.

**Qué no hace.**

- No modifica el grafo. La carga (`pipeline/load_graph.py`) no usa este módulo, y los merges del grafo los sigue decidiendo `domain/identity_resolution.py`.
- No impone coherencia entre pares. Las decisiones son por par: no hay transitividad (que $a$ se vincule con $b$ y $b$ con $c$ no obliga a vincular $a$ con $c$) ni restricción de que un registro se vincule con a lo sumo un registro de otra fuente.
- No usa casos etiquetados. Todo se estima sin supervisión, así que las probabilidades y las tasas de error que reporta son estimaciones que valen si el modelo es correcto, no mediciones.

Un vínculo falso que terminara en un merge haría desaparecer a una persona como entidad propia del registro. Por eso, donde el modelo tiene que elegir entre dos errores, elige no vincular (sección 6.5), y por eso este notebook intenta decir con precisión qué supuestos hace y dónde pueden fallar (sección 9).

**Cómo correrlo.** Desde la raíz del repo:

```
PYTHONPATH=src .venv/Scripts/python -m zona4_graph_loader.fellegi_sunter.correr
```

Opciones: `--mu` (por defecto 0.01), `--lambda` (0.01), `--inicios` (inicios aleatorios del EM además del inicio por defecto, 10), `--semilla` (0) y `--salida` (`data/processed/fellegi_sunter`). Escribe `pares.csv` (un par por fila) y `parametros.json` (agregados por estrato). La carpeta queda fuera de git, porque `pares.csv` tiene nombres de personas.

**Cómo usar este notebook para auditar.**

1. Cada sección define un paso con una fórmula y la acompaña con una o más celdas `contrato` que la ponen a prueba contra el código con ejemplos chicos.
2. Si cambia el código, corré `.venv/Scripts/python -m pytest -q tests/fellegi_sunter`. Si falla `test_contrato_notebook.py`, el texto y el código dicen cosas distintas: hay que decidir cuál está bien y corregir el otro.
3. Las secciones 7 y 8 leen las salidas de una corrida real y muestran sólo agregados, sin nombres ni claves. Sirven para revisar si los parámetros estimados tienen sentido y para reconstruir cada fila de `pares.csv` a partir de `parametros.json`.

La celda siguiente no es de contrato: sólo sirve para abrir el notebook en Jupyter. Busca la raíz del repo y agrega `src` al path de Python.

In [ ]:
import sys
from pathlib import Path

RAIZ = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pytest.ini").exists() and (p / "src" / "zona4_graph_loader").is_dir()),
    None,
)
if RAIZ is None:
    raise RuntimeError("Abrí el notebook desde una carpeta dentro del repo.")
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
print("raíz del repo:", RAIZ)

## 1. Campos, niveles y notación

### 1.1 Qué es un campo y qué es un nivel

El modelo nunca mira dos registros completos. Mira, uno por uno, algunos datos que las dos fuentes pueden tener en común. Cada uno de esos datos es un **campo**. Son cinco y se numeran con $k = 1, \dots, 5$:

| $k$ | campo | de dónde sale |
|---|---|---|
| 1 | apellido | la parte del nombre antes de la coma; en Parque de la Memoria, más los apellidos que sólo trae su nombre largo (sección 2.1) |
| 2 | nombre de pila | la parte del nombre después de la coma |
| 3 | fecha de nacimiento | el nodo de la persona o su arista `NACIO_EN` |
| 4 | fecha de secuestro | el nodo de la persona o su arista `SECUESTRADO_EN` |
| 5 | centro clandestino | los lugares de sus aristas `PRESENTE_EN` |

Para cada campo, comparar los valores de los dos registros da un **nivel**: una categoría que dice cuánto se parecen. Los niveles de un campo están ordenados del que indica más acuerdo (el nivel $0$) al que indica menos, y cada par cae en exactamente uno. Si el campo falta en alguno de los dos registros, el resultado no es un nivel sino $\varnothing$ (faltante).

| $k$ | campo | niveles, de más a menos acuerdo |
|---|---|---|
| 1, 2 | apellido, nombre de pila | `exacto`, `contenido` (uno tiene tokens de más), `errata` (corrigiendo errores de una letra en palabras de 4 letras o más, uno queda igual o contenido en el otro), `comparte` (algún token en común), `distinto` |
| 3, 4 | fechas | `exacta`, `dia_mes_invertidos`, `mismo_año_mes`, `mismo_año`, `distinta` |
| 5 | centro | `comparte` (algún centro en común), `ninguno_en_comun` |

Las condiciones exactas de cada nivel están en la sección 2. Acá alcanza con la idea.

Hay dos razones para usar niveles en lugar de los valores o de un puntaje continuo de similitud:

1. **El modelo necesita categorías.** $m_k(\ell)$ es la probabilidad de que un par de la misma persona caiga en el nivel $\ell$ del campo $k$, y $u_k(\ell)$ lo mismo para dos personas distintas. Con pocas categorías, esas probabilidades se estiman contando cuántos pares caen en cada una (sección 5).
2. **Los datos deciden cuánto vale cada nivel.** Cada nivel recibe su propio peso $w_k(\ell) = \log_2\big(m_k(\ell)/u_k(\ell)\big)$. No fijamos de antemano que "mismo año y mes" confirme o descarte: si la mayoría de esos pares resultan ser erratas de la misma persona, el modelo le da peso positivo; si no, negativo.

**Ejemplo.** Dos registros inventados (no son personas reales):

| | registro $a$ (Parque de la Memoria) | registro $b$ (MinJus víctimas) |
|---|---|---|
| nombre | Gómez, Ana María (nombre largo: Ana María Gómez Ruiz) | Gomes, Ana |
| fecha de nacimiento | | |
| fecha de secuestro | 4 de mayo de 1977 | 5 de abril de 1977 |
| centro | CCD X | CCD X |

Campo por campo:

- **Apellido.** $\{\text{gomez}, \text{ruiz}\}$ (el apellido corto, "Gómez", más "Ruiz", que el Parque sólo trae en el nombre largo) contra $\{\text{gomes}\}$. Ninguno contiene al otro, pero "gomes" es "gomez" con una letra cambiada; corregido, $\{\text{gomez}\}$ está contenido en el otro. Nivel `errata` ($\ell = 2$).
- **Nombre de pila.** $\{\text{ana}, \text{maria}\}$ contra $\{\text{ana}\}$: uno tiene un token de más. Nivel `contenido` ($\ell = 1$).
- **Fecha de nacimiento.** No está en ninguno de los dos: $\varnothing$.
- **Fecha de secuestro.** 1977-05-04 contra 1977-04-05: mismo año, día y mes intercambiados. Nivel `dia_mes_invertidos` ($\ell = 1$).
- **Centro.** Los dos pasaron por el CCD X. Nivel `comparte` ($\ell = 0$).

El par queda resumido en el vector de comparación, o patrón,

$$
\gamma(a, b) = (\gamma_1, \gamma_2, \gamma_3, \gamma_4, \gamma_5) = (2,\ 1,\ \varnothing,\ 1,\ 0).
$$

Eso es todo lo que el modelo sabe del par. Dos pares con el mismo patrón reciben la misma probabilidad, aunque los nombres sean otros. Por eso las definiciones de la sección 2 son el paso que más decide el resultado, y por eso conviene revisarlas con el equipo.

La celda siguiente verifica el ejemplo contra el código.

In [ ]:
# Contrato: el ejemplo de la sección 1.1.
from zona4_graph_loader.fellegi_sunter.comparaciones import NIVELES, parsear_fecha
from zona4_graph_loader.fellegi_sunter.pares import Registro, patron

a = Registro(
    clave="ejemplo:a", fuente="detalles_personas", roles=frozenset({"VICTIMA"}),
    apellido=frozenset({"gomez", "ruiz"}), nombre=frozenset({"ana", "maria"}),
    fecha_nacimiento=None, fecha_secuestro=parsear_fecha("1977-05-04"),
    centro=frozenset({"lugar:CCD:x"}), apellido_base=frozenset({"gomez"}),
)
b = Registro(
    clave="ejemplo:b", fuente="minjus_victimas", roles=frozenset({"VICTIMA"}),
    apellido=frozenset({"gomes"}), nombre=frozenset({"ana"}),
    fecha_nacimiento=None, fecha_secuestro=parsear_fecha("1977-04-05"),
    centro=frozenset({"lugar:CCD:x"}),
)
gamma = patron(a, b)
assert gamma == (2, 1, None, 1, 0), f"el texto dice (2, 1, ∅, 1, 0), el código da {gamma}"
assert NIVELES["apellido"][gamma[0]] == "errata"
assert NIVELES["nombre"][gamma[1]] == "contenido"
assert NIVELES["fecha_secuestro"][gamma[3]] == "dia_mes_invertidos"
assert NIVELES["centro"][gamma[4]] == "comparte"
assert patron(b, a) == gamma, "el patrón no depende del orden del par"
from zona4_graph_loader.fellegi_sunter.pares import pasa_filtro
assert pasa_filtro(a, b), "el apellido base, {gomez} contra {gomes}, pasa el filtro de la sección 3.2"


### 1.2 Notación

Esta sección fija los símbolos que se usan en todo el notebook y dice a qué objeto del código corresponde cada uno. Auditar es comparar fórmulas con código, así que conviene que la correspondencia sea explícita.

Un registro es una persona tal como la describe una fuente. Un par de registros de fuentes distintas se resume en un vector de comparación: un nivel de acuerdo por campo, o $\varnothing$ si el campo falta en alguno de los dos. El modelo trata cada par como si viniera de una de dos clases ocultas, misma persona o personas distintas, y estima la distribución de los niveles en cada clase.

| Símbolo | Qué es | En el código |
|---|---|---|
| $a, b$ | registros | `Registro` (`pares.py`) |
| $f(a)$ | fuente del registro $a$ | `Registro.fuente` |
| $\mathcal F$ | fuentes que entran al modelo | `FUENTES` |
| $s = \{f(a), f(b)\}$ | estrato: el par de fuentes del par $\{a, b\}$ | `estrato(a, b)` |
| $k = 1, \dots, K$, con $K = 5$ | campos, en este orden: apellido, nombre, fecha de nacimiento, fecha de secuestro, centro | `CAMPOS`; el campo $k$ es `CAMPOS[k-1]` |
| $x_k(a)$ | valor del campo $k$ en el registro $a$ | atributo de `Registro` |
| $x^{\mathrm{base}}_1(a)$ | apellido base: la parte del nombre del CDM antes de la coma; lo usa el filtro de pares | `Registro.apellido_filtro` (es `apellido_base` si hubo apellido extra, y `apellido` si no) |
| $t^{\mathrm{largo}}(a)$ | nombre largo del Parque de la Memoria, sin coma | `nombres_largos[clave]`, de `detalle.descripcion_nombre` |
| $e(a)$ | apellido extra: tokens del nombre largo que no están en el nombre corto | `apellido_extra(...)` |
| $x_1(a) = x^{\mathrm{base}}_1(a) \cup e(a)$ | apellido completo; es el que da el nivel $\gamma_1$ | `Registro.apellido` |
| $\gamma^{\mathrm{base}}_1(a, b)$ | nivel de la sección 2.2 aplicado a los apellidos base | `nivel_tokens(a.apellido_filtro, b.apellido_filtro)` |
| $L_k$ | cantidad de niveles del campo $k$ | `len(NIVELES[campo])` |
| $\ell \in \{0, \dots, L_k - 1\}$ | nivel de acuerdo; el 0 es el de más acuerdo | índice en `NIVELES[campo]` |
| $\varnothing$ | faltante | `FALTA`, que es `None` |
| $\gamma_k(a,b)$ | nivel del campo $k$ en el par | `nivel_tokens`, `nivel_fecha`, `nivel_centro` |
| $\gamma(a,b) = (\gamma_1, \dots, \gamma_K)$ | vector de comparación o patrón | `patron(a, b)` |
| $\Gamma$ | conjunto de patrones posibles | |
| $O(\gamma) = \{k : \gamma_k \neq \varnothing\}$ | campos observados del patrón | |
| $\mathcal P_s$ | pares que se comparan en el estrato $s$ (sección 3) | salida de `generar_pares` |
| $n_\gamma$, $N_s = \sum_\gamma n_\gamma$ | pares de $\mathcal P_s$ con patrón $\gamma$, y total | `conteos` |
| $C \in \{M, U\}$ | clase oculta del par: misma persona ($M$) o personas distintas ($U$) | |
| $\pi$ | $P(C = M)$ dentro de $\mathcal P_s$ | `Parametros.pi` |
| $m_k(\ell)$, $u_k(\ell)$ | $P(\gamma_k = \ell \mid M)$ y $P(\gamma_k = \ell \mid U)$ | `Parametros.m[k-1][ℓ]`, `Parametros.u[k-1][ℓ]` |
| $m(\gamma)$, $u(\gamma)$ | $\prod_{k \in O(\gamma)} m_k(\gamma_k)$ y lo mismo con $u$ | `_probabilidad` |
| $\theta = (\pi, m, u)$ | parámetros de un estrato | `Parametros` |
| $w_k(\ell) = \log_2 \big(m_k(\ell) / u_k(\ell)\big)$ | peso del campo $k$ en el nivel $\ell$ | `pesos_por_campo` |
| $w(\gamma)$ | peso del patrón, en bits | `peso` |
| $p(\gamma) = P(M \mid \gamma)$ | probabilidad posterior de misma persona | `probabilidad_match` |
| $r^{(t)}_\gamma$ | responsabilidad: $p(\gamma)$ evaluada en $\theta^{(t)}$ | `paso_e` |
| $\alpha$ | pseudo-conteo del paso M | `PSEUDO_CONTEO` |
| $\tau$ | largo mínimo de un token para considerar erratas | `LARGO_MINIMO_ERRATA` |
| $d_L(x, y)$ | distancia de Levenshtein entre dos cadenas | `_levenshtein_distance` |
| $\mu$, $\lambda$ | tasas de error objetivo | `--mu`, `--lambda` |
| $T_\mu$, $T_\lambda$ | umbrales sobre el peso | `Umbrales.t_mu`, `Umbrales.t_lambda` |

Convenciones:

- Todos los parámetros son de un estrato $s$. Escribimos $\pi$, $m_k$, $u_k$ en lugar de $\pi_s$, $m_{s,k}$, $u_{s,k}$ cuando no hay ambigüedad.
- Los campos se numeran desde 1 en el texto y desde 0 en el código. Los niveles se numeran desde 0 en los dos lados.
- Cada campo tiene $L_k + 1$ valores posibles en un patrón (sus niveles o $\varnothing$), así que $|\Gamma| = \prod_k (L_k + 1) = 6^4 \cdot 3 = 3888$.

In [ ]:
# Contrato: notación (sección 1).
from zona4_graph_loader.fellegi_sunter.comparaciones import CAMPOS, FALTA, NIVELES

assert CAMPOS == ("apellido", "nombre", "fecha_nacimiento", "fecha_secuestro", "centro"), (
    f"el orden de los campos k = 1..5 del texto no coincide con CAMPOS: {CAMPOS}"
)
K = len(CAMPOS)
assert K == 5, f"el texto usa K = 5, el código tiene {K} campos"
L = tuple(len(NIVELES[campo]) for campo in CAMPOS)
assert L == (5, 5, 5, 5, 2), f"el texto usa L = (5, 5, 5, 5, 2), el código tiene {L}"
tamano_gamma = 1
for L_k in L:
    tamano_gamma *= L_k + 1
assert tamano_gamma == 3888, tamano_gamma
assert FALTA is None, "el faltante ∅ es None en el código"

## 2. Niveles por campo

Esta sección define $\gamma_k(a, b)$ para cada campo. Es el paso que más decide el resultado: el modelo sólo ve niveles, nunca los valores originales, así que dos pares con el mismo patrón son indistinguibles para él. Las definiciones están en `comparaciones.py`.

Regla general: los niveles de un campo son excluyentes y se evalúan en el orden en que aparecen; el primero cuya condición se cumple es el nivel del par. En las definiciones por casos de abajo, cada línea vale sólo si no valió ninguna anterior.

Todos los niveles son simétricos: $\gamma_k(a, b) = \gamma_k(b, a)$. Las celdas de contrato lo verifican en cada ejemplo.

### 2.1 Normalización de nombres

El nombre de un registro viene como texto, en principio con la forma "Apellido(s), Nombre(s)". Se separa en la primera coma y cada parte se convierte en un conjunto de tokens con el mismo normalizador que usa la resolución de identidades (`name_token_set`).

Para un texto $t$, $\operatorname{tok}(t)$ se obtiene así: pasar a minúsculas, sacar acentos y diacríticos (descomposición NFKD; la ñ queda n), reemplazar `_` y todo carácter fuera de `[a-z0-9]` y espacios por un espacio, partir por espacios y descartar los tokens de una letra y los de la lista

$$
\mathcal V = \{\text{da, das, de, del, der, di, do, dos, el, filho, jr, junior, la, las, los, miss, mr, mrs, ms, van, y}\}.
$$

Con eso, el apellido base y el nombre de pila de un registro $a$ con nombre $t$ son

$$
\big(x^{\mathrm{base}}_1(a), x_2(a)\big) =
\begin{cases}
(\varnothing, \varnothing) & \text{si } t \text{ no tiene coma},\\
\big(\operatorname{tok}(t_{\text{antes}}),\ \operatorname{tok}(t_{\text{después}})\big) & \text{si la tiene},
\end{cases}
$$

donde $t_{\text{antes}}$ y $t_{\text{después}}$ son lo que queda antes y después de la primera coma, y un conjunto de tokens vacío se trata como $\varnothing$. Para tokens usamos $\varnothing$ tanto para "faltante" como para "conjunto vacío", porque el código los trata igual.

Sin coma no hay forma segura de saber qué tokens son apellido, así que los dos campos faltan. En la sección 3 se ve que un registro sin apellido no entra al modelo.

**Apellido completo en Parque de la Memoria.** En el CDM, el nombre de una ficha del Parque trae sólo el primer apellido ("Abad, Ana Catalina"), pero la ficha guarda también el nombre completo en `detalle.descripcion_nombre`, sin coma y con los nombres primero ("Ana Catalina Abad Scarlatta Perucca"). Como el nombre largo no separa nombres de apellidos, se toma como apellido todo token que no esté ya en el nombre corto:

$$
e(a) =
\begin{cases}
\operatorname{tok}\big(t^{\mathrm{largo}}(a)\big) \setminus \big(x^{\mathrm{base}}_1(a) \cup x_2(a)\big) & \text{si } a \text{ tiene nombre largo},\\
\varnothing & \text{si no},
\end{cases}
\qquad
x_1(a) = x^{\mathrm{base}}_1(a) \cup e(a),
$$

con $x_2(a) = \varnothing$ si falta el nombre de pila. En el ejemplo, $x^{\mathrm{base}}_1(a) = \{\text{abad}\}$, $x_2(a) = \{\text{ana}, \text{catalina}\}$, $e(a) = \{\text{scarlatta}, \text{perucca}\}$ y $x_1(a) = \{\text{abad}, \text{scarlatta}, \text{perucca}\}$. El nombre de pila no cambia.

Detalles:

- Sólo las fichas del Parque tienen nombre largo: `nombres_largos_parque` arma el mapa `registro:<n>` $\to$ `descripcion_nombre` a partir del archivo de detalles (`DETALLES_PATH`), y `correr.py` se lo pasa a `construir_registros`. El loader y el CDM no cambian.
- Es una unión: un token del nombre corto que no aparece en el largo se conserva.
- Un token del nombre de pila escrito distinto en el largo que en el corto ("Katalina" contra "Catalina") no está en $x_2(a)$ y se suma como apellido (sección 9).
- Las demás fuentes ya traen los dos apellidos antes de la coma: para ellas $e(a) = \varnothing$ y $x_1(a) = x^{\mathrm{base}}_1(a)$.

Al escribir esto (noviembre de 2026), de las 8932 fichas del Parque, 7840 sumaban al menos un token: 6823 uno, 973 dos, 43 tres y 1 cuatro. En 9 fichas algún token del nombre corto no aparecía en el largo. `parametros.json` guarda la cantidad por fuente en `registros_con_apellido_extra` (sección 7).

In [ ]:
# Contrato: normalización y separación de nombres (sección 2.1).
from zona4_graph_loader.domain.name_similarity import TRASH_TOKENS
from zona4_graph_loader.fellegi_sunter.comparaciones import separar_nombre


def T(*tokens):
    return frozenset(tokens)


assert TRASH_TOKENS == {
    "da", "das", "de", "del", "der", "di", "do", "dos", "el", "filho", "jr", "junior",
    "la", "las", "los", "miss", "mr", "mrs", "ms", "van", "y",
}, "la lista 𝒱 del texto no coincide con TRASH_TOKENS"

assert separar_nombre("Castiñeiras, José María") == (T("castineiras"), T("jose", "maria"))
assert separar_nombre("de la Fuente, Juan C.") == (T("fuente"), T("juan")), "partículas e iniciales"
assert separar_nombre("Toranzo - Palacín") == (None, None), "sin coma faltan los dos campos"
assert separar_nombre("Pérez,") == (T("perez"), None), "nombre vacío es faltante"
assert separar_nombre(", Juan") == (None, T("juan")), "apellido vacío es faltante"
assert separar_nombre("Pérez, Juan, Carlos") == (T("perez"), T("juan", "carlos")), "se corta en la primera coma"

In [ ]:
# Contrato: apellido completo del Parque (sección 2.1).
from zona4_graph_loader.fellegi_sunter.pares import apellido_extra, nombres_largos_parque

largos = nombres_largos_parque([
    {"registro": 1, "detalle": {"descripcion_nombre": "Ana Catalina Abad Scarlatta Perucca"}},
    {"registro": 2, "detalle": {}},                                  # sin nombre largo
    {"registro": None, "detalle": {"descripcion_nombre": "Otro Nombre"}},
])
assert largos == {"registro:1": "Ana Catalina Abad Scarlatta Perucca"}, largos

base, pila = separar_nombre("Abad, Ana Catalina")
assert (base, pila) == (T("abad"), T("ana", "catalina"))
assert apellido_extra(largos["registro:1"], base, pila) == T("scarlatta", "perucca"), "e(a) = tok(largo) − base − pila"
assert apellido_extra(None, base, pila) == frozenset(), "sin nombre largo, e(a) = ∅"
assert apellido_extra("Ana Catalina Abad", base, pila) == frozenset(), "el largo no trae nada nuevo"
assert apellido_extra("Abad Scarlatta", base, None) == T("scarlatta"), "sin nombre de pila se resta sólo el apellido"
# Un token del nombre de pila escrito distinto en el largo se suma como apellido (límite de la sección 9).
assert apellido_extra("Ana Katalina Abad Scarlatta", base, pila) == T("katalina", "scarlatta")

### 2.2 Apellido y nombre ($k = 1, 2$)

Los dos campos usan la misma comparación entre conjuntos de tokens $A = x_k(a)$ y $B = x_k(b)$. La idea: primero igualdad, después inclusión (una fuente registra un apellido o un nombre de más), después lo mismo pero perdonando erratas de una letra en tokens largos, después un token en común (también perdonando erratas).

**Errata.** Dos tokens $x$, $y$ son errata uno del otro si

$$
E(x, y) \iff x \neq y \ \wedge\ |x| \geq \tau \ \wedge\ |y| \geq \tau \ \wedge\ d_L(x, y) = 1, \qquad \tau = 4.
$$

La distancia de Levenshtein cuenta inserciones, borrados y sustituciones de una letra. Una transposición ("rosa" y "rsoa") está a distancia 2, así que no es errata. Entre tokens cortos ("li" y "lu") una letra de diferencia ya es otro nombre, y por eso se exige $\tau = 4$ en los dos.

**Corrección de erratas.** Antes de comparar por inclusión, cada conjunto se corrige mirando el otro. *Corregir $A$ hacia $B$* quiere decir recorrer los tokens de $A$ y hacer, con cada uno, lo siguiente:

1. si el token está en $B$, queda igual;
2. si no está en $B$ pero es errata de algún token de $B$, se reemplaza por ese token de $B$;
3. si no pasa ninguna de las dos cosas, queda igual.

El conjunto que resulta se escribe $c_B(A)$, "$A$ corregido hacia $B$". La corrección nunca agrega ni saca tokens que no tengan un parecido de una letra: sólo cambia la escritura de los que sí lo tienen.

Ejemplo, con $A = \{\text{gomes}, \text{ruiz}\}$ y $B = \{\text{gomez}\}$:

| token de $A$ | ¿está en $B$? | ¿es errata de un token de $B$? | queda |
|---|---|---|---|
| gomes | no | sí, de gomez | gomez |
| ruiz | no | no | ruiz |

Entonces $c_B(A) = \{\text{gomez}, \text{ruiz}\}$. Ahora $B$ está contenido en $c_B(A)$, y por eso el par cae en el nivel `errata` (definido abajo): es el caso de una fuente que escribió "Gomes Ruiz" y otra "Gómez".

Si un token es errata de más de un token de $B$, se elige el primero en orden alfabético. Por ejemplo, "perez" es errata de "peres" y de "perea"; corregido hacia $\{\text{peres}, \text{perea}\}$ queda "perea". El desempate sólo existe para que el resultado no dependa del orden en que vienen los datos; en los datos reales este caso es raro.

En fórmulas, para un token $x$:

$$
\varphi_B(x) =
\begin{cases}
x & \text{si } x \in B,\\
\text{el primero, en orden alfabético, de los } y \in B \text{ tales que } E(x, y) & \text{si hay alguno},\\
x & \text{si no hay ninguno},
\end{cases}
$$

y $c_B(A) = \{ \varphi_B(x) : x \in A \}$ aplica $\varphi_B$ a cada token de $A$. En el código es `_corregir_erratas(A, B)`.

**Nivel.**

$$
\gamma_k(a, b) =
\begin{cases}
\varnothing & \text{si } A = \varnothing \text{ o } B = \varnothing,\\
0 \ (\texttt{exacto}) & \text{si } A = B,\\
1 \ (\texttt{contenido}) & \text{si } A \subsetneq B \text{ o } B \subsetneq A,\\
2 \ (\texttt{errata}) & \text{si } c_B(A) \subseteq B \text{ o } B \subseteq c_B(A) \text{ o } c_A(B) \subseteq A \text{ o } A \subseteq c_A(B),\\
3 \ (\texttt{comparte}) & \text{si } c_B(A) \cap B \neq \varnothing \text{ o } c_A(B) \cap A \neq \varnothing,\\
4 \ (\texttt{distinto}) & \text{en otro caso}.
\end{cases}
$$

| $\ell$ | nivel | se cumple si (y no se cumplió uno anterior) | ejemplo |
|---|---|---|---|
| 0 | `exacto` | mismos tokens | {perez} y {perez} |
| 1 | `contenido` | uno tiene tokens de más | {godoy} y {godoy, ruiz} |
| 2 | `errata` | corrigiendo erratas de una letra, uno queda contenido en el otro | {castineira} y {castineyra}; {perez} y {peres, gomez} |
| 3 | `comparte` | al menos un token en común, igual o errata uno del otro | {juan, carlos} y {juan, pedro}; {perez, lopez} y {peres, gomez} |
| 4 | `distinto` | ninguno de los anteriores | {perez} y {gomez}; {li} y {lu} |

Observaciones que importan para leer los resultados:

- `errata` incluye combinaciones: {perez} y {peres, gomez} es errata más un token de más; {perez, gomez} y {peres, gomes} son dos erratas a la vez. Las dos caen en el nivel 2.
- `comparte` también perdona erratas: {perez, lopez} y {perez, gomez} es `comparte` por el token idéntico, y {perez, lopez} y {peres, gomez} es `comparte` porque "perez" y "peres" son errata uno del otro.
- **Caracterización del nivel 4.** $\gamma_k(a, b) = 4$ si y sólo si $A \cap B = \varnothing$ y no hay $x \in A$, $y \in B$ con $E(x, y)$. En efecto, $c_B(A) \cap B \neq \varnothing$ quiere decir que algún $x \in A$ cumple $\varphi_B(x) \in B$, y por la definición de $\varphi_B$ eso pasa si y sólo si $x \in B$ o $x$ es errata de algún token de $B$; lo mismo con $c_A(B) \cap A$. Entonces, si hay un token en común o un par de erratas, se cumple la condición del nivel 3 (y el nivel es el primero de 0 a 3 que se cumpla). Si no hay ninguno de los dos, no se cumple ninguna condición de 0 a 3: los niveles 0 y 1 implican $A \cap B \neq \varnothing$, y el 2 implica una corrección (punto siguiente).
- Si se llega al nivel 2, hubo al menos una corrección: si $c_B(A) = A$ y $c_A(B) = B$, la condición del nivel 2 dice $A \subseteq B$ o $B \subseteq A$, que ya habría dado nivel 0 o 1.
- La simetría de los niveles 2 y 3 sale de que sus condiciones incluyen las dos direcciones de corrección.

In [ ]:
# Contrato: la corrección de erratas (sección 2.2).
from zona4_graph_loader.fellegi_sunter.comparaciones import _corregir_erratas

T = lambda *tokens: frozenset(tokens)
# El ejemplo de la tabla.
assert _corregir_erratas(T("gomes", "ruiz"), T("gomez")) == T("gomez", "ruiz")
# Un token que ya está en B queda igual, aunque también sea errata de otro.
assert _corregir_erratas(T("peres"), T("peres", "perez")) == T("peres")
# Desempate alfabético: "perez" es errata de "peres" y de "perea".
assert _corregir_erratas(T("perez"), T("peres", "perea")) == T("perea")
# Sin parecido de una letra, nada cambia; tampoco entre tokens de menos de 4 letras.
assert _corregir_erratas(T("perez", "li"), T("gomez", "lu")) == T("perez", "li")


In [ ]:
# Contrato: errata y niveles de apellido y nombre (sección 2.2).
from zona4_graph_loader.fellegi_sunter.comparaciones import LARGO_MINIMO_ERRATA, nivel_tokens, son_errata

assert LARGO_MINIMO_ERRATA == 4, f"el texto usa τ = 4, el código {LARGO_MINIMO_ERRATA}"
assert NIVELES["apellido"] == ("exacto", "contenido", "errata", "comparte", "distinto")
assert NIVELES["nombre"] == NIVELES["apellido"]

# E(x, y)
assert son_errata("perez", "peres"), "sustitución"
assert son_errata("castineira", "castineiras"), "inserción"
assert son_errata("gomez", "gomz"), "borrado, el más corto tiene largo 4"
assert not son_errata("perez", "perez"), "x ≠ y"
assert not son_errata("lima", "lim"), "uno de los dos tiene menos de τ letras"
assert not son_errata("li", "lu"), "tokens cortos"
assert not son_errata("rosa", "rsoa"), "una transposición está a distancia 2"


def nivel_tok(a, b):
    i = nivel_tokens(a, b)
    assert nivel_tokens(b, a) == i, f"el nivel depende del orden: {sorted(a or ())} {sorted(b or ())}"
    return None if i is None else NIVELES["apellido"][i]


casos = [
    (T("perez"), T("perez"), "exacto"),
    (T("godoy"), T("godoy", "ruiz"), "contenido"),
    (T("castineira"), T("castineyra"), "errata"),
    (T("perez"), T("peres", "gomez"), "errata"),           # errata y un token de más
    (T("perez", "gomez"), T("peres", "gomes"), "errata"),  # dos correcciones
    (T("juan", "carlos"), T("juan", "karlos"), "errata"),  # gana errata sobre comparte
    (T("juan", "carlos"), T("juan", "pedro"), "comparte"),
    (T("perez", "lopez"), T("perez", "gomez"), "comparte"),
    (T("perez", "lopez"), T("peres", "gomez"), "comparte"),  # 'perez' y 'peres' son errata
    (T("perez"), T("gomez"), "distinto"),
    (T("li"), T("lu"), "distinto"),
    (T("lima"), T("lim"), "distinto"),
    (T("rosa"), T("rsoa"), "distinto"),
]
for a, b, esperado in casos:
    obtenido = nivel_tok(a, b)
    assert obtenido == esperado, f"{sorted(a)} vs {sorted(b)}: el texto dice {esperado}, el código da {obtenido}"

assert nivel_tok(None, T("perez")) is None
assert nivel_tok(T("perez"), frozenset()) is None, "un conjunto vacío es faltante"

### 2.3 Fechas de nacimiento y de secuestro ($k = 3, 4$)

Las fuentes traen fechas con distinta precisión: año, año y mes, o fecha completa. Se comparan hasta donde llegan las dos.

**Lectura.** Un valor de texto $v$, sin espacios al principio y al final, se convierte en una fecha $D = (y, \mathit{mes}, \mathit{dia})$:

$$
D(v) =
\begin{cases}
(y, \varnothing, \varnothing) & \text{si } v = \texttt{AAAA},\\
(y, \mathit{mes}, \varnothing) & \text{si } v = \texttt{AAAA-MM},\\
(y, \mathit{mes}, \mathit{dia}) & \text{si } v = \texttt{AAAA-MM-DD},\\
\varnothing & \text{para cualquier otro valor},
\end{cases}
$$

con exactamente cuatro, dos y dos dígitos, y siempre que sea una fecha del calendario: tiene que existir la fecha $(y, \mathit{mes}, \mathit{dia})$, completando con 1 el mes o el día que falten (se valida con `datetime.date`, que admite años de 1 a 9999). Centinelas como "DESCONOCIDA", fechas con barras o sin ceros a la izquierda, y fechas imposibles como "1976-13-45", "1976-02-30" o "1976-00" son faltantes. "1976-02-29" es válida (año bisiesto) y "1977-02-29" no.

**Nivel.** Para $D = (y, \mathit{mes}, \mathit{dia})$ y $D' = (y', \mathit{mes}', \mathit{dia}')$:

$$
\gamma_k(a, b) =
\begin{cases}
\varnothing & \text{si } D = \varnothing \text{ o } D' = \varnothing,\\
4 \ (\texttt{distinta}) & \text{si } y \neq y',\\
3 \ (\texttt{mismo\_año}) & \text{si } \mathit{mes} = \varnothing \text{ o } \mathit{mes}' = \varnothing,\\
0 \ (\texttt{exacta}) & \text{si } \mathit{dia} \neq \varnothing,\ \mathit{dia}' \neq \varnothing \text{ y } D = D',\\
1 \ (\texttt{dia\_mes\_invertidos}) & \text{si } \mathit{dia} \neq \varnothing,\ \mathit{dia}' \neq \varnothing,\ \mathit{mes} = \mathit{dia}' \text{ y } \mathit{dia} = \mathit{mes}',\\
2 \ (\texttt{mismo\_año\_mes}) & \text{si } \mathit{mes} = \mathit{mes}',\\
3 \ (\texttt{mismo\_año}) & \text{en otro caso}.
\end{cases}
$$

| $\ell$ | nivel | ejemplo |
|---|---|---|
| 0 | `exacta` | 1976-03-24 y 1976-03-24 |
| 1 | `dia_mes_invertidos` | 1976-04-05 y 1976-05-04 |
| 2 | `mismo_año_mes` | 1976-03-24 y 1976-03-25; 1948-07 y 1948-07-17 |
| 3 | `mismo_año` | 1976-03-24 y 1976-08-24; 1948 y 1948-07-17 |
| 4 | `distinta` | 1976-03-24 y 1977-03-24 |

Consecuencias de la precisión parcial:

- Si una de las dos fechas sólo trae el año, el mejor nivel posible es `mismo_año`; si una sólo trae año y mes, el mejor es `mismo_año_mes`. Dos fechas "1948" y "1948" son `mismo_año`, nunca `exacta`.
- Por eso los niveles 2 y 3 mezclan dos situaciones: acuerdo parcial entre fechas completas y acuerdo total entre fechas imprecisas. El modelo les asigna una sola probabilidad por clase. Como la precisión depende sobre todo de la fuente y los parámetros se estiman por par de fuentes (sección 5.9), la mezcla es parcial, pero existe.
- `dia_mes_invertidos` exige las dos fechas completas. Si día y mes coinciden (03-03), la fecha es `exacta` porque ese nivel se evalúa antes.
- El año distinto manda: "1976-03-24" y "1977" es `distinta`.

In [ ]:
# Contrato: fechas (sección 2.3).
from zona4_graph_loader.fellegi_sunter.comparaciones import Fecha, nivel_fecha, parsear_fecha

assert NIVELES["fecha_nacimiento"] == ("exacta", "dia_mes_invertidos", "mismo_año_mes", "mismo_año", "distinta")
assert NIVELES["fecha_secuestro"] == NIVELES["fecha_nacimiento"]

assert parsear_fecha("1976-03-24") == Fecha(1976, 3, 24)
assert parsear_fecha(" 1948-07 ") == Fecha(1948, 7), "se sacan los espacios de los bordes"
assert parsear_fecha("1948") == Fecha(1948)
for valor in ("DESCONOCIDA", "", None, "24/03/1976", "1976-3-24", "76-03-24"):
    assert parsear_fecha(valor) is None, f"{valor!r} tendría que ser faltante"
for valor in ("1976-13-45", "1976-02-30", "1976-00", "1977-02-29", "0000"):
    assert parsear_fecha(valor) is None, f"{valor!r} no es una fecha del calendario: tendría que ser faltante"
assert parsear_fecha("1976-02-29") == Fecha(1976, 2, 29), "1976 es bisiesto"


def nivel_f(a, b):
    i = nivel_fecha(parsear_fecha(a), parsear_fecha(b))
    assert nivel_fecha(parsear_fecha(b), parsear_fecha(a)) == i, f"el nivel depende del orden: {a} {b}"
    return None if i is None else NIVELES["fecha_secuestro"][i]


casos = [
    ("1976-03-24", "1976-03-24", "exacta"),
    ("1976-04-05", "1976-05-04", "dia_mes_invertidos"),
    ("1976-03-24", "1976-03-25", "mismo_año_mes"),
    ("1976-03-24", "1976-08-24", "mismo_año"),
    ("1976-03-24", "1977-03-24", "distinta"),
    ("1976-03-24", "1977", "distinta"),           # el año distinto manda
    ("1948", "1948-07-17", "mismo_año"),
    ("1948", "1948", "mismo_año"),               # dos años iguales nunca son 'exacta'
    ("1948-07", "1948-07-17", "mismo_año_mes"),
    ("1948-07", "1948-07", "mismo_año_mes"),
    ("1948-07", "1948-08-17", "mismo_año"),
    ("1976-03-03", "1976-03-03", "exacta"),       # día = mes: 'exacta' se evalúa antes
    ("1976-04-05", "1976-05", "mismo_año"),      # invertidos exige las dos completas
    ("1976-04-05", "1976-05-05", "mismo_año"),
]
for a, b, esperado in casos:
    obtenido = nivel_f(a, b)
    assert obtenido == esperado, f"{a} vs {b}: el texto dice {esperado}, el código da {obtenido}"

assert nivel_f(None, "1976") is None and nivel_f("DESCONOCIDA", "1976-03-24") is None

### 2.4 Centro ($k = 5$)

El campo centro de un registro es el conjunto $S(a)$ de lugares (`lugar_key`) unidos a la persona por una arista `PRESENTE_EN`, o $\varnothing$ si no tiene ninguna. Se cuenta cualquier lugar con esa arista, no sólo los que son centros clandestinos.

$$
\gamma_5(a, b) =
\begin{cases}
\varnothing & \text{si } S(a) = \varnothing \text{ o } S(b) = \varnothing,\\
0 \ (\texttt{comparte}) & \text{si } S(a) \cap S(b) \neq \varnothing,\\
1 \ (\texttt{ninguno\_en\_comun}) & \text{en otro caso}.
\end{cases}
$$

Un registro sin lugares no dice "no estuvo en ningún centro": dice que la fuente no lo registra. Por eso es faltante y no nivel 1.

In [ ]:
# Contrato: centro (sección 2.4).
from zona4_graph_loader.fellegi_sunter.comparaciones import nivel_centro

assert NIVELES["centro"] == ("comparte", "ninguno_en_comun")
assert NIVELES["centro"][nivel_centro(T("lugar:a", "lugar:b"), T("lugar:b"))] == "comparte"
assert NIVELES["centro"][nivel_centro(T("lugar:a"), T("lugar:b"))] == "ninguno_en_comun"
assert nivel_centro(None, T("lugar:a")) is None
assert nivel_centro(frozenset(), T("lugar:a")) is None, "sin lugares es faltante, no 'ninguno_en_comun'"

## 3. Registros y pares que se comparan

Esta sección define qué registros entran y cuáles de sus pares se comparan (`pares.py`). Importa para auditar por dos razones: todo par que no se compara se trata como "no es la misma persona" sin que el modelo lo mire, y los parámetros estimados describen sólo a los pares que sí se comparan.

### 3.1 Registros

Entran las personas de estas fuentes, antes de cualquier resolución de identidades:

$$
\mathcal F = \{\texttt{detalles\_personas},\ \texttt{archivo\_memoria},\ \texttt{minjus\_victimas},\ \texttt{minjus\_imputados},\ \texttt{juicios\_condenados}\}.
$$

Abuelas (`nietxs_relacion`) queda afuera: su campo `nombre` es el identificador del caso ("Apellido1 - Apellido2"), no el nombre de una persona. Además, un registro entra sólo si tiene apellido base, es decir, si su nombre tiene coma y la parte de antes de la coma deja al menos un token (sección 2.1):

$$
\mathcal R = \{ a : f(a) \in \mathcal F,\ x^{\mathrm{base}}_1(a) \neq \varnothing \}.
$$

Como $x_1(a) \supseteq x^{\mathrm{base}}_1(a)$, el apellido completo tampoco falta, y el apellido nunca falta en un par del modelo: $1 \in O(\gamma)$ para todo patrón. El apellido completo $x_1(a)$ se arma como en la sección 2.1; sólo difiere del base en las fichas del Parque con nombre largo.

**De dónde salen las fechas.** Para cada campo de fecha se usa el valor del nodo y, si el nodo no tiene valor, el de la arista correspondiente (`NACIO_EN` para nacimiento, `SECUESTRADO_EN` para secuestro). Son las funciones `_fechas_de` y `_fechas_de_aristas` de `identity_resolution.py`, las mismas que usa la resolución de identidades:

$$
v_k(a) =
\begin{cases}
\text{valor del nodo} & \text{si el nodo tiene un valor no vacío},\\
\text{fecha de la arista} & \text{si no, y las aristas de ese tipo de } a \text{ traen una única fecha completa } \texttt{AAAA-MM-DD},\\
\varnothing & \text{en otro caso},
\end{cases}
\qquad x_k(a) = D\big(v_k(a)\big).
$$

Detalles a tener en cuenta: de las aristas sólo se toman fechas completas; si una persona tiene dos aristas del mismo tipo con fechas distintas, ese campo no se toma de las aristas; y un valor no vacío en el nodo gana aunque no sea una fecha (en ese caso queda $\varnothing$ aunque la arista tenga una fecha). En los datos del repo, al escribir este notebook, las fechas de `detalles_personas` llegan por las aristas y las de las otras cuatro fuentes por el nodo.

**Centro y roles.** El centro es el conjunto de `lugar_key` de las aristas `PRESENTE_EN` (sección 2.4). Los roles son los de la persona en su fuente.

In [ ]:
# Contrato: qué registros entran y de dónde salen las fechas (sección 3.1).
from zona4_graph_loader.fellegi_sunter.pares import FUENTES, construir_registros

assert FUENTES == (
    "detalles_personas", "archivo_memoria", "minjus_victimas", "minjus_imputados", "juicios_condenados",
), f"las fuentes del texto no coinciden con FUENTES: {FUENTES}"
assert "nietxs_relacion" not in FUENTES


def persona(clave, nombre, fuente, roles=("VICTIMA",), **campos):
    return {"persona_key": clave, "nombre": nombre, "fuente": fuente, "roles": list(roles), **campos}


def arista(clave, tipo, fecha=None, lugar="lugar:x"):
    return {"persona_key": clave, "lugar_key": lugar, "tipo_relacion": tipo, "fecha": fecha}


dataset = {
    "personas": [
        persona("p:1", "Pérez, Juan", "detalles_personas"),
        persona("p:2", "Pérez, Juan", "minjus_victimas", fecha_secuestro="1976-05"),
        persona("p:3", "Pérez, Juan", "archivo_memoria", fecha_secuestro="1976-05-04"),
        persona("p:4", "Toranzo - Palacín", "nietxs_relacion", roles=("NIETX",)),
        persona("p:5", "Sin Coma Juan", "detalles_personas"),
        persona("p:6", ", Juan", "detalles_personas"),
        persona("p:7", "Pérez, Juan", "otra_fuente"),
    ],
    "eventos_espaciales": [
        arista("p:1", "SECUESTRADO_EN", "1976-05-04"),
        arista("p:1", "NACIO_EN", "1950-01-02", lugar="lugar:y"),
        arista("p:1", "NACIO_EN", "1950-01-03", lugar="lugar:z"),   # dos fechas distintas
        arista("p:1", "PRESENTE_EN", lugar="lugar:ccd"),
        arista("p:2", "NACIO_EN", "1950"),                          # incompleta
        arista("p:3", "SECUESTRADO_EN", "1977-01-01"),              # el nodo gana
    ],
}
registros = {r.clave: r for r in construir_registros(dataset)}
assert set(registros) == {"p:1", "p:2", "p:3"}, (
    f"entran sólo fuentes de 𝓕 con apellido; entraron {sorted(registros)}"
)
assert registros["p:1"].fecha_secuestro == Fecha(1976, 5, 4), "sin valor en el nodo se usa la arista"
assert registros["p:1"].fecha_nacimiento is None, "dos aristas con fechas distintas: el campo falta"
assert registros["p:1"].centro == frozenset({"lugar:ccd"}), "centro = lugares de PRESENTE_EN"
assert registros["p:2"].fecha_secuestro == Fecha(1976, 5), "el nodo puede traer precisión parcial"
assert registros["p:2"].fecha_nacimiento is None, "de las aristas sólo se toman fechas completas"
assert registros["p:2"].centro is None, "sin PRESENTE_EN el centro falta"
assert registros["p:3"].fecha_secuestro == Fecha(1976, 5, 4), "si el nodo tiene valor, gana"

### 3.2 Las tres condiciones

Un par $\{a, b\} \subseteq \mathcal R$ se compara si y sólo si cumple las tres condiciones:

$$
\mathcal P = \Big\{ \{a, b\} \subseteq \mathcal R :\ f(a) \neq f(b),\ \ \neg \operatorname{inc}(a, b),\ \ \gamma^{\mathrm{base}}_1(a, b) \neq 4 \Big\},
\qquad
\mathcal P_s = \big\{ \{a, b\} \in \mathcal P : \{f(a), f(b)\} = s \big\}.
$$

1. **Fuentes distintas.** Una fuente no se compara consigo misma, igual que en la resolución de identidades. El modelo no busca duplicados dentro de una fuente.
2. **Roles compatibles.** Con $\operatorname{vic}(a) \iff \text{roles}(a) \cap \{\texttt{VICTIMA}, \texttt{NIETX}\} \neq \varnothing$ y $\operatorname{rep}(a) \iff \text{roles}(a) \cap \{\texttt{REPRESOR}, \texttt{COMPLICE}\} \neq \varnothing$,
$$
\operatorname{inc}(a, b) = \big(\operatorname{vic}(a) \wedge \operatorname{rep}(b)\big) \vee \big(\operatorname{rep}(a) \wedge \operatorname{vic}(b)\big).
$$
Un registro sin roles es compatible con cualquiera. Es `_roles_incompatibles` de `identity_resolution.py`.
3. **Filtro del apellido base.** El nivel de la sección 2.2 aplicado a los apellidos base no es `distinto` (`pasa_filtro`). Por la caracterización del nivel 4, eso equivale a que los apellidos base compartan un token o tengan dos tokens, de largo al menos $\tau$, a distancia de Levenshtein 1:
$$
\gamma^{\mathrm{base}}_1(a, b) \neq 4 \iff x^{\mathrm{base}}_1(a) \cap x^{\mathrm{base}}_1(b) \neq \varnothing \ \vee\ \exists\, x \in x^{\mathrm{base}}_1(a),\ y \in x^{\mathrm{base}}_1(b) : E(x, y).
$$

Todo par que no cumple la condición 3 queda fuera del modelo y se trata como "no es la misma persona". [Sadinle (2018), sección 5.2.1](https://doi.org/10.1214/18-AOAS1178) hace algo parecido: fija como no coreferentes todos los pares que caen en el peor nivel de desacuerdo del nombre de pila o del apellido (sección verificada en la versión de arXiv, https://arxiv.org/abs/1812.09590v1). Acá sólo se mira el apellido.

**Por qué el filtro usa el apellido base y el nivel el completo.** Son dos usos distintos del apellido.

- El filtro decide qué pares entran al modelo, y con eso la población sobre la que se estiman $\pi$, $m$ y $u$. Con el apellido base, el filtro es el mismo que antes de sumar el nombre largo: entran exactamente los mismos pares. Si usara el apellido completo, entrarían todos los pares que sólo comparten el segundo apellido del Parque (una ficha "Scarlatta, Ana" de otra fuente contra la del Parque "Abad" con extra "Scarlatta"), casi todos de personas distintas.
- El nivel $\gamma_1$ usa el apellido completo para distinguir. Contra la ficha del Parque "Abad" con extra "Scarlatta": "Abad Scarlatta" pasa de `contenido` a `exacto`; "Abad Coronel" pasa de `contenido` a `comparte`; "Abad" solo pasa de `exacto` a `contenido`.

**Proposición.** Si $\{a, b\} \in \mathcal P$, entonces $\gamma_1(a, b) \neq 4$: el nivel `distinto` del apellido completo nunca aparece en los pares del modelo.

*Prueba.* Por la condición 3 y la caracterización del nivel 4 (sección 2.2), hay un token $t \in x^{\mathrm{base}}_1(a) \cap x^{\mathrm{base}}_1(b)$ o un par $x \in x^{\mathrm{base}}_1(a)$, $y \in x^{\mathrm{base}}_1(b)$ con $E(x, y)$. Como $x^{\mathrm{base}}_1 \subseteq x_1$ en los dos registros, el mismo $t$ está en $x_1(a) \cap x_1(b)$, o el mismo par $(x, y)$ está en $x_1(a) \times x_1(b)$. Por la caracterización otra vez, $\gamma_1(a, b) \neq 4$. $\square$

La recíproca no vale: la ficha "Scarlatta, Ana" tiene $\gamma_1 = 1$ (`contenido`) contra la del Parque, pero $\gamma^{\mathrm{base}}_1 = 4$ y no entra.

### 3.3 El índice y por qué no pierde pares

Comparar todos contra todos es cuadrático. `generar_pares` usa un índice por token de apellido: sólo mira los pares que comparten alguna clave y después confirma con el nivel real. Para un token $t$,

$$
\kappa(t) =
\begin{cases}
\{t\} \cup \{\, t_{-i} : i = 1, \dots, |t| \,\} & \text{si } |t| \geq \tau,\\
\{t\} & \text{si no},
\end{cases}
\qquad
\kappa(a) = \bigcup_{t \in x^{\mathrm{base}}_1(a)} \kappa(t),
$$

donde $t_{-i}$ es $t$ sin su letra $i$. Se miran los pares con $\kappa(a) \cap \kappa(b) \neq \varnothing$ y, de esos, se conservan los que cumplen las tres condiciones.

El índice usa el apellido base, el mismo del filtro.

**Lema.** Si $\gamma^{\mathrm{base}}_1(a, b) \neq 4$, entonces $\kappa(a) \cap \kappa(b) \neq \varnothing$. Por lo tanto el índice no pierde ningún par de $\mathcal P$, y como cada candidato se confirma con el nivel real, tampoco agrega ninguno.

*Prueba.* Sean $A = x^{\mathrm{base}}_1(a)$ y $B = x^{\mathrm{base}}_1(b)$, ambos no vacíos. Por la caracterización de la sección 2.2, si el nivel no es 4 hay un token en común o un par de erratas. Si hay un token $t \in A \cap B$, entonces $t \in \kappa(t)$ de los dos lados. Si no, existen $x \in A$, $y \in B$ con $E(x, y)$, o sea $|x|, |y| \geq \tau$ y $d_L(x, y) = 1$. Si la diferencia es una sustitución en la posición $i$, entonces $x_{-i} = y_{-i}$ está en $\kappa(x) \cap \kappa(y)$. Si es una inserción, digamos $|x| = |y| + 1$, borrar de $x$ la letra sobrante da $y$, y $y \in \kappa(x) \cap \kappa(y)$. $\square$

La prueba usa que los dos tokens tienen largo al menos $\tau$, que es justamente la condición de $E$; por eso el índice genera borrados sólo para tokens de largo $\geq \tau$.

### 3.4 Consecuencias del filtro

- **Nombre igual y apellido base distinto queda afuera por construcción.** Por ejemplo, una mujer registrada con el apellido del marido en una fuente y con el propio en otra, apellidos con dos letras de diferencia o con una transposición ("rosa" y "rsoa"), o un apellido de menos de $\tau$ letras con una errata ("Paz" y "Pas"; también "lima" y "lim"). Tampoco entra un registro que sólo comparte con una ficha del Parque un apellido que el Parque trae en el nombre largo. Esos pares no se comparan y el modelo no puede recuperarlos.
- **El nivel `distinto` del apellido nunca aparece en $\mathcal P$** (proposición de la sección 3.2). Sus $m_1(4)$ y $u_1(4)$ se estiman sólo con el pseudo-conteo (sección 5.3), nunca entran en el peso de ningún par y no hay que interpretarlos.
- **El filtro cambia $\pi$ y $m_1, u_1$, y la independencia condicional se conserva bajo un supuesto.** Sea $F$ el evento "el par cumple la condición 3". Supongamos que el modelo de la sección 4 vale para todos los pares de un estrato, con parámetros $\pi^\circ, m^\circ, u^\circ$, y que además los demás campos son independientes, dada la clase, de toda la información de apellido del par: $(\gamma_2, \dots, \gamma_K) \perp\!\!\!\perp (\gamma_1, F) \mid C$. Entonces $P(\gamma \mid C, F) = P(\gamma_1 \mid C, F) \prod_{k \geq 2} P(\gamma_k \mid C)$, y en $\mathcal P_s$ vale el mismo modelo con
$$
m_1(\ell) = P(\gamma_1 = \ell \mid M, F), \qquad m_1(4) = 0, \qquad m_k = m^\circ_k \ (k \geq 2), \qquad
\pi = \frac{\pi^\circ P(F \mid M)}{\pi^\circ P(F \mid M) + (1 - \pi^\circ) P(F \mid U)},
$$
y lo mismo para $u$. En un par sin fichas del Parque con nombre largo, $F$ es $\{\gamma_1 \neq 4\}$, una función de $\gamma_1$, y el supuesto adicional se cumple solo: $m_1(\ell) = m^\circ_1(\ell) / (1 - m^\circ_1(4))$ para $\ell < 4$. Con el apellido completo del Parque, $F$ depende del apellido base y no sólo de $\gamma_1$, y el supuesto pide que esa diferencia no tenga que ver con los otros campos. En cualquier caso, $\pi$ es la proporción de pares de la misma persona dentro de los pares filtrados, mucho mayor que la proporción entre todos los pares del estrato.

In [ ]:
# Contrato: las tres condiciones y el índice (secciones 3.2 y 3.3).
import itertools

from zona4_graph_loader.fellegi_sunter.pares import _claves_de_indice, estrato, generar_pares, pasa_filtro, patron

regs = construir_registros({"personas": [
    persona("d:1", "Pérez, Juan", "detalles_personas"),
    persona("d:2", "Pérez, Juan", "detalles_personas"),                      # misma fuente que d:1
    persona("v:1", "Peres, Juan", "minjus_victimas"),
    persona("c:1", "Pérez, Juan", "juicios_condenados", roles=("REPRESOR",)),
    persona("i:1", "Pérez, Juan", "minjus_imputados", roles=("REPRESOR",)),
    persona("a:1", "Gómez, Juan", "archivo_memoria"),                        # nombre igual, apellido distinto
]})
obtenidos = {tuple(sorted((a.clave, b.clave))): g for a, b, g in generar_pares(regs)}
assert set(obtenidos) == {("d:1", "v:1"), ("d:2", "v:1"), ("c:1", "i:1")}, sorted(obtenidos)
assert obtenidos[("d:1", "v:1")] == (2, 0, None, None, None), "apellido errata, nombre exacto, el resto falta"
assert estrato(regs[2], regs[0]) == ("detalles_personas", "minjus_victimas"), "el estrato es el par ordenado de fuentes"

# Lema de la sección 3.3, exhaustivo sobre cadenas de 3 a 5 letras en {a, b}.
tokens = ["".join(p) for n in (3, 4, 5) for p in itertools.product("ab", repeat=n)]
for x, y in itertools.combinations(tokens, 2):
    if son_errata(x, y):
        assert _claves_de_indice(x) & _claves_de_indice(y), f"{x} y {y} son errata y no comparten clave"

# El índice da exactamente los mismos pares que comparar todos contra todos.
nombres = [
    "Castiñeira, Ana", "Castineyra, Ana", "Godoy Ruiz, Luis", "Godoy, Luis", "Ruiz, Pedro",
    "Li, Wen", "Lu, Wen", "Perez, Juan", "Peres Gomez, Juan", "Gomes, Juan", "Diaz, Rosa",
    "Dias, Rosa", "Perez Lopez, Juan", "Paz, Ana", "Pas, Ana", "Rodrigues Paz, Ana",
]
fuentes = ["detalles_personas", "minjus_victimas", "archivo_memoria"]
regs = construir_registros({"personas": [
    persona(f"k:{i}", n, f) for i, (n, f) in enumerate(itertools.product(nombres, fuentes))
]})
fuerza_bruta = {
    (a.clave, b.clave) for a, b in itertools.combinations(regs, 2)
    if a.fuente != b.fuente and pasa_filtro(a, b)
}
con_indice = {(a.clave, b.clave) for a, b, _ in generar_pares(regs)}
assert con_indice == fuerza_bruta, "el índice pierde o agrega pares"
assert all(g[0] != NIVELES["apellido"].index("distinto") for _, _, g in generar_pares(regs)), (
    "el nivel 'distinto' del apellido nunca aparece en los pares del modelo"
)

In [ ]:
# Contrato: filtro sobre el apellido base y nivel sobre el completo (sección 3.2).
import random

from zona4_graph_loader.fellegi_sunter.pares import Registro

parque = {"personas": [
    persona("registro:1", "Abad, Ana Catalina", "detalles_personas"),
    persona("v:1", "Abad Scarlatta, Ana Catalina", "minjus_victimas"),
    persona("v:2", "Abad Coronel, Ana Catalina", "minjus_victimas"),
    persona("v:3", "Abad, Ana Catalina", "minjus_victimas"),
    persona("m:1", "Scarlatta, Ana", "archivo_memoria"),
]}
largos = nombres_largos_parque([{"registro": 1, "detalle": {"descripcion_nombre": "Ana Catalina Abad Scarlatta"}}])
con_largo = {r.clave: r for r in construir_registros(parque, largos)}
sin_largo = {r.clave: r for r in construir_registros(parque)}

p1 = con_largo["registro:1"]
assert p1.apellido == T("abad", "scarlatta") and p1.apellido_base == T("abad") and p1.apellido_filtro == T("abad")
assert con_largo["v:1"].apellido_base is None and con_largo["v:1"].apellido_filtro == T("abad", "scarlatta")


def niveles_apellido(registros):
    return {tuple(sorted((a.clave, b.clave))): NIVELES["apellido"][g[0]] for a, b, g in generar_pares(list(registros.values()))}


con, sin = niveles_apellido(con_largo), niveles_apellido(sin_largo)
assert set(con) == set(sin), "el filtro usa el apellido base: entran los mismos pares con y sin nombre largo"
for par, antes, ahora in [
    (("registro:1", "v:1"), "contenido", "exacto"),
    (("registro:1", "v:2"), "contenido", "comparte"),
    (("registro:1", "v:3"), "exacto", "contenido"),
]:
    assert (sin[par], con[par]) == (antes, ahora), (par, sin[par], con[par])
assert ("m:1", "registro:1") not in con, "sólo comparte el segundo apellido del Parque: no entra"
assert NIVELES["apellido"][nivel_tokens(p1.apellido, con_largo["m:1"].apellido)] == "contenido", (
    "la recíproca de la proposición no vale: el apellido completo no es 'distinto', pero el base sí"
)

# Proposición de la sección 3.2 e índice, con apellidos extra sorteados.
rng = random.Random(3)
vocab = ["perez", "peres", "gomez", "gomes", "ruiz", "paz", "pas", "abad", "abat", "lopez", "coronel"]
fuentes = ["detalles_personas", "minjus_victimas", "archivo_memoria"]
regs = []
for i in range(60):
    base = frozenset(rng.sample(vocab, rng.randint(1, 2)))
    extra = frozenset(rng.sample(vocab, rng.randint(0, 2))) - base
    regs.append(Registro(f"r:{i}", fuentes[i % 3], frozenset({"VICTIMA"}), base | extra, T("ana"),
                         None, None, None, apellido_base=base if extra else None))


def comparte_o_errata(A, B):
    return bool(A & B) or any(son_errata(x, y) for x in A for y in B)


esperado = set()
for a, b in itertools.combinations(regs, 2):
    assert pasa_filtro(a, b) == comparte_o_errata(a.apellido_filtro, b.apellido_filtro), "caracterización del filtro"
    if a.fuente != b.fuente and pasa_filtro(a, b):
        esperado.add((a.clave, b.clave))
pares_rand = list(generar_pares(regs))
assert {(a.clave, b.clave) for a, b, _ in pares_rand} == esperado, "el índice sobre el apellido base pierde o agrega pares"
assert any(a.apellido_base is not None or b.apellido_base is not None for a, b, _ in pares_rand)
assert all(g[0] != NIVELES["apellido"].index("distinto") for _, _, g in pares_rand), "proposición de la sección 3.2"

## 4. El modelo y el peso

Esta sección define el modelo probabilístico de los patrones, el peso de un par y su probabilidad posterior (`em.py`). Es lo que convierte un patrón en un número; auditarlo es poder recalcular ese número a mano.

### 4.1 Mezcla de dos clases con independencia condicional

En cada estrato $s$, cada par de $\mathcal P_s$ pertenece a una clase oculta $C \in \{M, U\}$, con $P(C = M) = \pi$. Dada la clase, los campos son independientes y cada uno sigue una distribución categórica sobre sus niveles:

$$
P(\gamma \mid M) = m(\gamma) = \prod_{k \in O(\gamma)} m_k(\gamma_k),
\qquad
P(\gamma \mid U) = u(\gamma) = \prod_{k \in O(\gamma)} u_k(\gamma_k),
$$

$$
P(\gamma) = \pi\, m(\gamma) + (1 - \pi)\, u(\gamma),
\qquad \sum_{\ell = 0}^{L_k - 1} m_k(\ell) = \sum_{\ell = 0}^{L_k - 1} u_k(\ell) = 1.
$$

Es el modelo de [Fellegi y Sunter (1969)](https://doi.org/10.1080/01621459.1969.10501049) con niveles de acuerdo parcial, estimado con EM como en [Winkler (1988)](https://www.census.gov/library/working-papers/2000/adrm/rr2000-05.html) (el link lleva a la versión revisada que el U.S. Census Bureau publicó en 2000 como informe RR2000/05).

### 4.2 Por qué el producto recorre sólo los campos observados

Sea $\gamma^\ast$ el patrón completo (el que se vería si ningún campo faltara) y $R \subseteq \{1, \dots, K\}$ el conjunto de campos observados, de modo que lo que se ve es $(\gamma^\ast_R, R)$. Por independencia condicional, sumar sobre los valores de los campos faltantes da

$$
P(\gamma^\ast_R \mid C = M) = \sum_{\gamma^\ast_{\bar R}} \prod_{k=1}^K m_k(\gamma^\ast_k) = \prod_{k \in R} m_k(\gamma^\ast_k) \prod_{k \notin R} \underbrace{\sum_{\ell} m_k(\ell)}_{=1} = \prod_{k \in R} m_k(\gamma^\ast_k),
$$

y lo mismo con $u$. Falta justificar que se puede ignorar el mecanismo que decide $R$. El supuesto es que qué campos faltan no depende de la clase ni de los valores que faltan:

$$
P(R \mid C, \gamma^\ast) = P(R \mid \gamma^\ast_R) \quad \text{para toda clase } C \text{ y todo } \gamma^\ast_{\bar R},
\qquad \text{o sea} \qquad R \perp\!\!\!\perp (C, \gamma^\ast_{\bar R}) \mid \gamma^\ast_R .
$$

No alcanza con "faltantes al azar dada la clase": si $P(R \mid C, \gamma^\ast)$ dependiera de $C$, aparecería un factor distinto en cada término de la mezcla. La cuenta: la distribución de lo observado es

$$
P(\gamma^\ast_R, R) = \sum_{c \in \{M, U\}} P(C = c)\, P(\gamma^\ast_R \mid c)\, P(R \mid \gamma^\ast_R, c),
$$

si $R$ no depende de los valores faltantes dada la clase. Si además $P(R \mid \gamma^\ast_R, c)$ no depende de la clase $c$, que es la otra mitad del supuesto, sale del paréntesis:

$$
P(\gamma^\ast_R, R) = P(R \mid \gamma^\ast_R)\, \big[\pi\, m(\gamma) + (1 - \pi)\, u(\gamma)\big],
$$

y el primer factor no depende de $\theta$, así que no cambia el máximo de la verosimilitud. Es el supuesto de faltantes al azar (MAR) de [Rubin (1976)](https://doi.org/10.1093/biomet/63.3.581), contando a la clase $C$ entre lo no observado. En palabras: qué campos faltan puede depender de lo que se ve, pero no de si el par es de la misma persona ni de los valores que faltan.

Que la clase cuente es lo delicado. En este repo casi todo faltante es estructural: depende de qué campos registra cada fuente, y eso es constante dentro de un estrato, así que es ignorable. Pero dentro de una fuente, si las personas mejor documentadas (con fechas) aparecen también en más fuentes, la presencia de un campo depende de la clase y el supuesto falla. Por la misma cuenta, la razón $m(\gamma)/u(\gamma)$ que se usa como peso es la razón de verosimilitud de lo observado, $(\gamma^\ast_R, R)$, sólo bajo este supuesto: si no, faltaría el factor $P(R \mid \gamma^\ast_R, M) / P(R \mid \gamma^\ast_R, U)$.

### 4.3 Verosimilitud y conteos por patrón

Tratando los pares de $\mathcal P_s$ como independientes, la log-verosimilitud es

$$
\ell(\theta) = \sum_{\{a,b\} \in \mathcal P_s} \ln P\big(\gamma(a,b)\big) = \sum_{\gamma \in \Gamma} n_\gamma \ln\big(\pi\, m(\gamma) + (1 - \pi)\, u(\gamma)\big).
$$

Depende de los datos sólo a través de los conteos $(n_\gamma)_{\gamma \in \Gamma}$, así que por el teorema de factorización son estadísticos suficientes para $\theta$. Por eso el EM trabaja con un diccionario patrón $\to$ conteo, que tiene a lo sumo $4 \cdot 6^3 \cdot 3 = 2592$ entradas (el apellido nunca falta ni es `distinto`, secciones 3.1 y 3.2) aunque los pares sean decenas de miles.

La independencia entre pares es falsa: un mismo registro aparece en muchos pares, y si $a$ es la misma persona que $b$, no lo es de otro registro $b'$ de la misma fuente (si la fuente no tiene duplicados). $\ell(\theta)$ es entonces una pseudo-verosimilitud, la usual en el enfoque de [Fellegi y Sunter (1969)](https://doi.org/10.1080/01621459.1969.10501049); [Sadinle (2017)](https://doi.org/10.1080/01621459.2016.1148612) muestra cómo imponer que el emparejamiento entre dos fuentes sea uno a uno.

### 4.4 Peso y probabilidad posterior

El peso de un patrón es el logaritmo en base 2 de la razón de verosimilitud. Por la independencia condicional se descompone en una suma de pesos por campo, sólo sobre los campos observados:

$$
w(\gamma) = \log_2 \frac{m(\gamma)}{u(\gamma)} = \sum_{k \in O(\gamma)} w_k(\gamma_k),
\qquad
w_k(\ell) = \log_2 \frac{m_k(\ell)}{u_k(\ell)}.
$$

Un campo faltante aporta 0 bits, ni a favor ni en contra. Por la regla de Bayes,

$$
p(\gamma) = P(M \mid \gamma) = \frac{\pi\, m(\gamma)}{\pi\, m(\gamma) + (1 - \pi)\, u(\gamma)},
$$

y tomando logit, $\operatorname{logit} x = \ln \frac{x}{1 - x}$,

$$
\operatorname{logit} p(\gamma) = \ln \frac{\pi}{1 - \pi} + \ln \frac{m(\gamma)}{u(\gamma)} = \operatorname{logit} \pi + w(\gamma) \ln 2 .
$$

Cada bit de peso duplica las chances $p / (1 - p)$. Con $w = 0$ la posterior es igual a $\pi$; con $\pi$ chico, hace falta mucho peso para llegar a una posterior alta. Las decisiones de la sección 6 dependen sólo de $w$, porque $p$ es una función creciente de $w$ dentro de un estrato.

In [ ]:
# Contrato: peso y posterior calculados a mano (sección 4.4).
import math

from zona4_graph_loader.fellegi_sunter.em import Parametros, peso, pesos_por_campo, probabilidad_match

# Parámetros inventados, en el orden de CAMPOS. Cada fila suma 1.
ejemplo = Parametros(
    pi=0.05,
    m=[[0.6, 0.3, 0.05, 0.04, 0.01], [0.8, 0.05, 0.05, 0.05, 0.05], [0.7, 0.05, 0.1, 0.1, 0.05],
       [0.6, 0.05, 0.15, 0.1, 0.1], [0.8, 0.2]],
    u=[[0.1, 0.4, 0.2, 0.29, 0.01], [0.01, 0.01, 0.01, 0.07, 0.9], [0.001, 0.002, 0.007, 0.05, 0.94],
       [0.005, 0.001, 0.03, 0.3, 0.664], [0.1, 0.9]],
)
for dist in (ejemplo.m, ejemplo.u):
    assert all(math.isclose(sum(fila), 1.0) for fila in dist)

# Apellido contenido, nombre exacto, sin fecha de nacimiento, secuestro mismo año y mes, sin centro.
gamma = (1, 0, None, 2, None)
# w = log2(0.3/0.4) + log2(0.8/0.01) + log2(0.15/0.03) = log2(0.75 · 80 · 5) = log2(300)
assert math.isclose(peso(gamma, ejemplo), math.log2(300)), peso(gamma, ejemplo)
por_campo = pesos_por_campo(gamma, ejemplo)
assert por_campo[2] is None and por_campo[4] is None, "un campo faltante no tiene peso"
assert math.isclose(sum(w for w in por_campo if w is not None), peso(gamma, ejemplo))

# p = π m(γ) / (π m(γ) + (1 − π) u(γ)), con m(γ) = 0.3 · 0.8 · 0.15 y u(γ) = 0.4 · 0.01 · 0.03
p_mano = 0.05 * 0.036 / (0.05 * 0.036 + 0.95 * 0.00012)
assert math.isclose(p_mano, 15 / 15.95)
assert math.isclose(probabilidad_match(gamma, ejemplo), p_mano)
# logit p = logit π + w ln 2
logit = lambda x: math.log(x / (1 - x))
assert math.isclose(logit(probabilidad_match(gamma, ejemplo)), logit(0.05) + math.log2(300) * math.log(2))

# Sin campos observados, w = 0 y la posterior es π.
vacio = (None,) * 5
assert peso(vacio, ejemplo) == 0 and math.isclose(probabilidad_match(vacio, ejemplo), 0.05)

## 5. Estimación con EM

Esta sección deriva el algoritmo que estima $\theta = (\pi, m, u)$ en cada estrato y dice qué garantiza y qué no. Para auditar importan tres cosas: que cada paso del código sea la fórmula derivada, que el objetivo no baje, y que la solución elegida no dependa de por dónde se empezó.

### 5.1 Objetivo: máximo a posteriori con pseudo-conteo

Si un nivel no aparece en una clase, su estimador de máxima verosimilitud es 0 y el peso de ese nivel es $\pm\infty$. Para evitarlo, el código suma un pseudo-conteo $\alpha$ a cada nivel. Eso equivale a maximizar la densidad a posteriori con una previa $\operatorname{Dirichlet}(1 + \alpha, \dots, 1 + \alpha)$ independiente en cada $m_k$ y cada $u_k$, y sin previa (o con previa plana) en $\pi$. El objetivo, sin la constante de normalización de la previa, es

$$
J(\theta) = \sum_{\gamma} n_\gamma \ln\big(\pi\, m(\gamma) + (1 - \pi)\, u(\gamma)\big) + \alpha \sum_{k=1}^K \sum_{\ell=0}^{L_k - 1} \big(\ln m_k(\ell) + \ln u_k(\ell)\big),
\qquad \alpha = 1,
$$

porque el logaritmo de la densidad $\operatorname{Dirichlet}(\beta)$ es $\sum_\ell (\beta_\ell - 1) \ln p_\ell$ más una constante. Es la función `objetivo`.

### 5.2 Paso E

Con datos completos (la clase $z_{ab} \in \{0, 1\}$ de cada par), la log-verosimilitud completa es

$$
\ell_c(\theta) = \sum_{\gamma} \sum_{\text{pares con patrón } \gamma} \Big[ z \big(\ln \pi + \textstyle\sum_{k \in O(\gamma)} \ln m_k(\gamma_k)\big) + (1 - z)\big(\ln(1 - \pi) + \textstyle\sum_{k \in O(\gamma)} \ln u_k(\gamma_k)\big) \Big].
$$

Es lineal en $z$, así que su esperanza dada la información observada y $\theta^{(t)}$ se obtiene reemplazando $z$ por

$$
r^{(t)}_\gamma = E\big[z \mid \gamma, \theta^{(t)}\big] = \frac{\pi^{(t)} m^{(t)}(\gamma)}{\pi^{(t)} m^{(t)}(\gamma) + (1 - \pi^{(t)})\, u^{(t)}(\gamma)},
$$

que depende del par sólo por su patrón. Queda

$$
Q(\theta \mid \theta^{(t)}) = \sum_{\gamma} n_\gamma \Big[ r^{(t)}_\gamma \big(\ln \pi + \textstyle\sum_{k \in O(\gamma)} \ln m_k(\gamma_k)\big) + (1 - r^{(t)}_\gamma)\big(\ln(1 - \pi) + \textstyle\sum_{k \in O(\gamma)} \ln u_k(\gamma_k)\big) \Big].
$$

### 5.3 Paso M

Se maximiza $Q(\theta \mid \theta^{(t)}) + \alpha \sum_{k, \ell} (\ln m_k(\ell) + \ln u_k(\ell))$. La función separa en $\pi$, en cada $m_k$ y en cada $u_k$. Para $\pi$, derivar $\sum_\gamma n_\gamma [r_\gamma \ln \pi + (1 - r_\gamma) \ln (1 - \pi)]$ da

$$
\pi^{(t+1)} = \frac{1}{N_s} \sum_\gamma n_\gamma\, r^{(t)}_\gamma .
$$

Para $m_k$, la parte que depende de él es $\sum_\ell c_{k\ell} \ln m_k(\ell)$ con $c_{k\ell} = \alpha + \sum_{\gamma : \gamma_k = \ell} n_\gamma r^{(t)}_\gamma$. Con un multiplicador de Lagrange para $\sum_\ell m_k(\ell) = 1$ sale $m_k(\ell) \propto c_{k\ell}$:

$$
m^{(t+1)}_k(\ell) = \frac{\alpha + \sum_{\gamma : \gamma_k = \ell} n_\gamma\, r^{(t)}_\gamma}{L_k\, \alpha + \sum_{\gamma : \gamma_k \neq \varnothing} n_\gamma\, r^{(t)}_\gamma},
\qquad
u^{(t+1)}_k(\ell) = \frac{\alpha + \sum_{\gamma : \gamma_k = \ell} n_\gamma\, (1 - r^{(t)}_\gamma)}{L_k\, \alpha + \sum_{\gamma : \gamma_k \neq \varnothing} n_\gamma\, (1 - r^{(t)}_\gamma)} .
$$

Cada par cuenta como $r_\gamma$ en $M$ y $1 - r_\gamma$ en $U$, y para el campo $k$ sólo cuentan los pares que lo tienen observado. Para $\pi$ cuentan todos.

Dos casos particulares que se ven en los resultados:

- **Apellido `distinto`.** Ningún par de $\mathcal P_s$ tiene $\gamma_1 = 4$ (proposición de la sección 3.2) y el apellido nunca falta, así que $m_1(4) = \alpha / (L_1 \alpha + \pi N_s)$ y $u_1(4) = \alpha / (L_1 \alpha + (1 - \pi) N_s)$: sólo el pseudo-conteo. Los otros cuatro niveles de $m_1$ suman $1 - m_1(4)$, apenas menos que 1.
- **Campo nunca observado en el estrato.** Si $\gamma_k = \varnothing$ en todos los pares, $c_{k\ell} = \alpha$ para todo $\ell$ y desde la primera iteración $m_k(\ell) = u_k(\ell) = 1 / L_k$. Ese campo no aporta nada: no aparece en ningún peso (siempre falta) y tampoco aportaría si apareciera ($w_k = 0$).

In [ ]:
# Contrato: pasos E y M e inicio por defecto, calculados a mano (secciones 5.1 a 5.3 y 5.6).
from zona4_graph_loader.fellegi_sunter.em import (
    PSEUDO_CONTEO, inicio_por_defecto, objetivo, paso_e, paso_m,
)

assert PSEUDO_CONTEO == 1.0, f"el texto usa α = 1, el código {PSEUDO_CONTEO}"

# Dos campos, de 2 y 3 niveles; un patrón tiene el segundo campo faltante.
niveles = [2, 3]
conteos = {(0, 0): 30, (1, 2): 60, (0, None): 10}
r = {(0, 0): 0.9, (1, 2): 0.1, (0, None): 0.5}   # responsabilidades cualesquiera
nuevo = paso_m(conteos, r, niveles)

# π = Σ n r / N = (27 + 6 + 5) / 100
assert math.isclose(nuevo.pi, 0.38)
# Campo 1, observado en los 100 pares: Σ n r = 38, Σ n (1 − r) = 62.
assert all(map(math.isclose, nuevo.m[0], [(1 + 27 + 5) / (2 + 38), (1 + 6) / (2 + 38)]))
assert all(map(math.isclose, nuevo.u[0], [(1 + 3 + 5) / (2 + 62), (1 + 54) / (2 + 62)]))
# Campo 2, observado en 90 pares: Σ n r = 33, Σ n (1 − r) = 57; el nivel 1 sólo tiene el pseudo-conteo.
assert all(map(math.isclose, nuevo.m[1], [(1 + 27) / (3 + 33), 1 / (3 + 33), (1 + 6) / (3 + 33)]))
assert all(map(math.isclose, nuevo.u[1], [(1 + 3) / (3 + 57), 1 / (3 + 57), (1 + 54) / (3 + 57)]))

# Paso E: r = P(M | γ) con los parámetros actuales.
e = paso_e(conteos, nuevo)
assert math.isclose(e[(0, None)], 0.38 * (33 / 40) / (0.38 * (33 / 40) + 0.62 * (9 / 64)))

# Objetivo: log-verosimilitud más α Σ ln m + α Σ ln u.
J = sum(n * math.log(nuevo.pi * nuevo.m[0][g[0]] * (nuevo.m[1][g[1]] if g[1] is not None else 1)
                     + (1 - nuevo.pi) * nuevo.u[0][g[0]] * (nuevo.u[1][g[1]] if g[1] is not None else 1))
        for g, n in conteos.items())
J += PSEUDO_CONTEO * sum(math.log(p) for dist in (nuevo.m, nuevo.u) for fila in dist for p in fila)
assert math.isclose(objetivo(conteos, nuevo), J)

# Inicio por defecto (sección 5.6): π = 0.1, m geométrico, u = frecuencias observadas + α.
inicio = inicio_por_defecto(conteos, niveles)
assert inicio.pi == 0.1
assert all(map(math.isclose, inicio.m[0], [2 / 3, 1 / 3]))
assert all(map(math.isclose, inicio.m[1], [4 / 7, 2 / 7, 1 / 7]))
assert all(map(math.isclose, inicio.u[0], [41 / 102, 61 / 102]))
assert all(map(math.isclose, inicio.u[1], [31 / 93, 1 / 93, 61 / 93]))
# Con otro pseudo-conteo, el inicio usa ese valor y no la constante.
inicio_2 = inicio_por_defecto(conteos, niveles, pseudo=2.0)
assert all(map(math.isclose, inicio_2.u[0], [42 / 104, 62 / 104]))

### 5.4 Por qué el objetivo no baja

Sea $\log P(\gamma; \theta) = Q_\gamma(\theta \mid \theta') - H_\gamma(\theta \mid \theta')$, donde $Q_\gamma$ es la esperanza de la log-verosimilitud completa de un par con patrón $\gamma$ y $H_\gamma(\theta \mid \theta') = E_{\theta'}[\ln P(z \mid \gamma; \theta) \mid \gamma]$. Sumando con pesos $n_\gamma$ y agregando la log-previa $\ln g(\theta)$:

$$
J(\theta) = Q(\theta \mid \theta') + \ln g(\theta) - H(\theta \mid \theta').
$$

Por la desigualdad de Gibbs, $H(\theta \mid \theta') \leq H(\theta' \mid \theta')$ para todo $\theta$. Como el paso M elige $\theta^{(t+1)}$ que maximiza $Q(\cdot \mid \theta^{(t)}) + \ln g$,

$$
J(\theta^{(t+1)}) - J(\theta^{(t)}) = \underbrace{\big[Q(\theta^{(t+1)} \mid \theta^{(t)}) + \ln g(\theta^{(t+1)})\big] - \big[Q(\theta^{(t)} \mid \theta^{(t)}) + \ln g(\theta^{(t)})\big]}_{\geq 0} + \underbrace{H(\theta^{(t)} \mid \theta^{(t)}) - H(\theta^{(t+1)} \mid \theta^{(t)})}_{\geq 0} \geq 0 .
$$

Es el argumento de [Dempster, Laird y Rubin (1977)](https://doi.org/10.1111/j.2517-6161.1977.tb01600.x), con la log-previa sumada a $Q$. Garantiza que $J$ no baja, no que se llegue al máximo global: el EM puede quedarse en un máximo local o en un punto de ensilladura. Por eso se corre desde varios inicios (sección 5.6).

### 5.5 Criterio de parada

`ajustar_em` itera hasta que el mayor cambio absoluto entre dos iteraciones consecutivas, sobre todos los parámetros, sea menor que la tolerancia, o hasta un máximo de iteraciones:

$$
\max\Big( |\pi^{(t)} - \pi^{(t-1)}|,\ \max_{k, \ell} |m^{(t)}_k(\ell) - m^{(t-1)}_k(\ell)|,\ \max_{k, \ell} |u^{(t)}_k(\ell) - u^{(t-1)}_k(\ell)| \Big) < 10^{-10},
\qquad t \leq 5000 .
$$

Si se corta por el máximo, `convergio` queda en falso y así aparece en `parametros.json`. El código guarda $J(\theta^{(t)})$ de cada iteración en `historia`, empezando por el inicio.

### 5.6 Inicios

- **Inicio por defecto**, determinista: $\pi^{(0)} = 0.1$; en $m$, cada nivel con la mitad de probabilidad que el anterior, $m^{(0)}_k(\ell) = 2^{-\ell} / \sum_{j=0}^{L_k - 1} 2^{-j}$; en $u$, la frecuencia observada de cada nivel con el pseudo-conteo, $u^{(0)}_k(\ell) = (\alpha + \sum_{\gamma : \gamma_k = \ell} n_\gamma) / (L_k \alpha + \sum_{\gamma : \gamma_k \neq \varnothing} n_\gamma)$, porque casi todos los pares son de personas distintas.
- **Inicios aleatorios**: $\pi^{(0)} \sim \operatorname{Uniforme}(0.01, 0.5)$ y cada $m^{(0)}_k$ y $u^{(0)}_k$ con distribución $\operatorname{Dirichlet}(1, \dots, 1)$ (uniforme en el simplex), independientes, sorteados con `random.Random(semilla)`.

`ajustar_em_varios_inicios` corre el EM desde el inicio por defecto y desde `inicios_aleatorios` inicios sorteados, y se queda con el de mayor $J$. Devuelve también todos los ajustes, para ver si distintos inicios llegan a la misma solución. Que lleguen al mismo $J$ y al mismo $\pi$ es una señal (no una prueba) de que no hay máximos locales competidores.

### 5.7 El problema de las etiquetas

$J$ no cambia si se intercambian las clases: $(\pi, m, u) \mapsto (1 - \pi, u, m)$ deja igual cada término $\pi m(\gamma) + (1 - \pi) u(\gamma)$, y la previa es la misma para $m$ y $u$. El EM puede entonces terminar con la clase de "misma persona" en el lugar de $U$. `ordenar_etiquetas`, aplicada al final de cada ajuste, llama $M$ a la clase en la que el nivel 0 (el de más acuerdo) es más probable, sumando sobre los campos:

$$
\Delta(\theta) = \sum_{k=1}^K \big(m_k(0) - u_k(0)\big);
\qquad
\text{si } \Delta(\theta) < 0, \text{ se reemplaza } (\pi, m, u) \text{ por } (1 - \pi, u, m).
$$

Con $\Delta = 0$ no se cambia nada. Un campo nunca observado aporta 0 a $\Delta$. Es una regla razonable, no una garantía: si en un estrato los pares de la misma persona tuvieran el nivel 0 menos frecuente que los de personas distintas en la mayoría de los campos, la regla elegiría mal. Por eso conviene revisar las tablas de la sección 7.

In [ ]:
# Contrato: el EM recupera parámetros conocidos, el objetivo no baja y las etiquetas (secciones 5.4 a 5.7).
import random
from collections import Counter

from zona4_graph_loader.fellegi_sunter.em import ajustar_em, ajustar_em_varios_inicios, ordenar_etiquetas

# Tres campos (de 3, 3 y 2 niveles), el mínimo para que la mezcla sea identificable (sección 5.8).
NIV = [3, 3, 2]
VERDAD = Parametros(
    pi=0.2,
    m=[[0.8, 0.15, 0.05], [0.7, 0.2, 0.1], [0.9, 0.1]],
    u=[[0.1, 0.3, 0.6], [0.05, 0.15, 0.8], [0.2, 0.8]],
)


def simular(params, n, rng):
    conteos = Counter()
    for _ in range(n):
        dist = params.m if rng.random() < params.pi else params.u
        conteos[tuple(rng.choices(range(len(fila)), weights=fila)[0] for fila in dist)] += 1
    return conteos


sim = simular(VERDAD, 30_000, random.Random(7))
ajuste = ajustar_em(sim, NIV)
assert ajuste.convergio, "el EM no convergió"
assert abs(ajuste.pi - VERDAD.pi) < 0.02, ajuste.pi
for k in range(len(NIV)):
    for l in range(NIV[k]):
        assert abs(ajuste.m[k][l] - VERDAD.m[k][l]) < 0.03, ("m", k, l, ajuste.m[k][l])
        assert abs(ajuste.u[k][l] - VERDAD.u[k][l]) < 0.03, ("u", k, l, ajuste.u[k][l])

# Sección 5.4: J no baja en ninguna iteración.
h = ajuste.historia
assert all(b >= a - 1e-9 for a, b in zip(h, h[1:])), "el objetivo bajó en alguna iteración"
assert math.isclose(ajuste.objetivo, h[-1])

# Sección 5.7: J es invariante al intercambio de clases, y ordenar_etiquetas lo deshace.
intercambiado = Parametros(pi=1 - ajuste.pi, m=ajuste.u, u=ajuste.m)
assert math.isclose(objetivo(sim, intercambiado), objetivo(sim, ajuste), rel_tol=1e-12)
reordenado = ordenar_etiquetas(intercambiado)
assert math.isclose(reordenado.pi, ajuste.pi) and reordenado.m == ajuste.m
# Empezando con las clases al revés, el resultado igual llama M a la clase de más acuerdo.
al_reves = ajustar_em(sim, NIV, inicio=Parametros(pi=0.8, m=[list(f) for f in VERDAD.u], u=[list(f) for f in VERDAD.m]))
assert sum(fm[0] - fu[0] for fm, fu in zip(al_reves.m, al_reves.u)) >= 0 and al_reves.pi < 0.5

# Sección 5.3: un campo nunca observado queda en 1/L_k y no cambia el resto.
con_campo_vacio = Counter({g + (None,): n for g, n in sim.items()})
ajuste_4 = ajustar_em(con_campo_vacio, NIV + [4])
assert ajuste_4.m[3] == [0.25] * 4 and ajuste_4.u[3] == [0.25] * 4
assert math.isclose(ajuste_4.pi, ajuste.pi, rel_tol=1e-9)

# Sección 5.6: varios inicios; se elige el de mayor J y todos quedan con las etiquetas ordenadas.
mejor, ajustes = ajustar_em_varios_inicios(sim, NIV, inicios_aleatorios=3, semilla=1)
assert len(ajustes) == 4, "inicio por defecto + 3 aleatorios"
assert mejor.objetivo == max(a.objetivo for a in ajustes)
assert all(sum(fm[0] - fu[0] for fm, fu in zip(a.m, a.u)) >= 0 for a in ajustes)

### 5.8 Identificabilidad

Que el EM converja no alcanza: hace falta que los parámetros estén determinados por la distribución de los patrones. Una mezcla de dos clases con independencia condicional y $K$ variables categóricas es un modelo de clases latentes.

Con $K = 2$ campos no es identificable. Con dos campos binarios, por ejemplo, hay cinco parámetros ($\pi$, $m_1(0)$, $m_2(0)$, $u_1(0)$, $u_2(0)$) y la tabla de $2 \times 2$ tiene tres grados de libertad. Con tres campos binarios hay siete parámetros y siete grados de libertad, así que la cuenta ya no lo descarta. [Allman, Matias y Rhodes (2009)](https://doi.org/10.1214/09-AOS689) estudian la identificabilidad de modelos de clases latentes con muchas variables observadas, con un teorema de Kruskal sobre descomposiciones de tensores de tres vías en el centro de sus resultados. La consecuencia que usamos acá es que, con dos clases y al menos tres variables de al menos dos niveles, los parámetros son genéricamente identificables salvo el intercambio de etiquetas. Esa consecuencia la derivamos de la condición de Kruskal para parámetros genéricos ($\min(r, L_1) + \min(r, L_2) + \min(r, L_3) \geq 2r + 2$ con $r = 2$ clases, que da $6 \geq 6$); no citamos un teorema numerado del artículo porque no lo verificamos en el texto.

"Genéricamente" quiere decir salvo un conjunto de parámetros de medida cero. Ese conjunto incluye los casos que importan en la práctica: un campo con $m_k = u_k$ no informa sobre la clase y no cuenta. Para este modelo la consecuencia es:

- En cada estrato hacen falta al menos tres campos observados e informativos, y suficientes pares que los observen a la vez. Un campo nunca observado en el estrato (sección 5.3) no cuenta.
- Con exactamente tres, el modelo está en el límite: es identificable en teoría, pero la estimación puede ser inestable.
- El apellido cuenta como un campo con cuatro niveles efectivos (el quinto no aparece en $\mathcal P_s$, sección 3.2).

La sección 7 cuenta, para cada estrato, cuántos campos fueron observados.

### 5.9 Estimación por estrato

$\theta$ se estima por separado en cada par de fuentes $s$ (`correr.ajustar_estratos`). Las razones:

- La distribución de los niveles entre pares de la misma persona depende de cómo registra cada fuente. Si una fuente guarda los dos apellidos y la otra uno, `contenido` es el nivel típico de $M$ en ese estrato; en otro estrato puede ser `exacto`.
- Qué campos existen depende de las fuentes: hay estratos sin fecha de nacimiento o sin centro.
- $\pi$ depende del solapamiento entre las dos fuentes y de cuántos pares deja el filtro.

Juntar estratos con parámetros distintos no da un modelo de dos clases con independencia condicional: el estrato actuaría como una variable oculta más, que induce dependencia entre campos dentro de cada clase. Como el estrato es observable, se condiciona en él. El costo es tener menos pares por estrato.

En cada estrato, $\pi$ es la proporción de pares de la misma persona dentro de $\mathcal P_s$, los pares filtrados (sección 3.4), no entre todos los pares posibles. $\pi N_s$ es el número esperado de pares de la misma persona en $\mathcal P_s$.

### 5.10 Resumen de lo que garantiza el EM

- $J$ no baja (5.4) y la corrida termina por tolerancia o por máximo de iteraciones (5.5).
- No garantiza el máximo global; los inicios múltiples lo mitigan (5.6).
- Las etiquetas se fijan con una regla heurística (5.7).
- La identificabilidad requiere al menos tres campos informativos observados (5.8).
- Los parámetros describen los pares filtrados de un estrato (5.9) y valen bajo independencia condicional y faltantes ignorables (4.2).

## 6. Regla de decisión

Esta sección define cómo se pasa del peso a una decisión (`decision.py`): el resultado de Fellegi y Sunter, cómo el código estima las tasas de error, y las convenciones para los casos que la teoría no resuelve. Es la parte que más directamente se traduce en errores sobre personas.

### 6.1 El resultado de Fellegi y Sunter

Una regla de decisión $d$ asigna a cada patrón $\gamma$ probabilidades $d(A_1 \mid \gamma)$, $d(A_2 \mid \gamma)$, $d(A_3 \mid \gamma)$ de las tres acciones: $A_1$ vincular, $A_2$ revisar (revisión humana), $A_3$ no vincular. Sus dos tasas de error son

$$
\mu(d) = P(A_1 \mid U) = \sum_{\gamma \in \Gamma} u(\gamma)\, d(A_1 \mid \gamma),
\qquad
\lambda(d) = P(A_3 \mid M) = \sum_{\gamma \in \Gamma} m(\gamma)\, d(A_3 \mid \gamma),
$$

la tasa de vínculos falsos y la de vínculos perdidos. [Fellegi y Sunter (1969)](https://doi.org/10.1080/01621459.1969.10501049) muestran que, si se ordenan los patrones por razón de verosimilitud $m(\gamma)/u(\gamma)$ decreciente, la regla que vincula los patrones de arriba hasta acumular $\mu$ de masa en $U$, descarta los de abajo hasta acumular $\lambda$ de masa en $M$ (aleatorizando en los patrones de borde para llegar exactamente a $\mu$ y $\lambda$) y manda a revisión el resto, minimiza la probabilidad de revisar entre todas las reglas con $\mu(d) \leq \mu$ y $\lambda(d) \leq \lambda$. Vale cuando las dos zonas no se superponen.

Es un test de razón de verosimilitud con dos umbrales. Un argumento al estilo Neyman–Pearson lo explica: para minimizar $P(A_2 \mid U) = 1 - \mu(d) - P(A_3 \mid U)$ con $P(A_3 \mid M) \leq \lambda$ conviene poner $A_3$ en los patrones de menor razón; para minimizar $P(A_2 \mid M) = 1 - \lambda(d) - P(A_1 \mid M)$ con $P(A_1 \mid U) \leq \mu$ conviene poner $A_1$ en los de mayor razón. Si las dos zonas no se superponen, la misma regla logra las dos cosas a la vez. Como $w = \log_2 (m / u)$ es creciente en la razón, se puede escribir con umbrales sobre $w$:

$$
d(\gamma) =
\begin{cases}
A_1 \ (\text{vincular}) & \text{si } w(\gamma) \geq T_\mu,\\
A_2 \ (\text{revisar}) & \text{si } T_\lambda < w(\gamma) < T_\mu,\\
A_3 \ (\text{no vincular}) & \text{si } w(\gamma) \leq T_\lambda .
\end{cases}
$$

La optimalidad vale con $m$ y $u$ verdaderos. Con $m$ y $u$ estimados, vale sólo en la medida en que el modelo sea correcto.

### 6.2 Cómo estima el código las tasas

El código no suma $u(\gamma)$ sobre todo $\Gamma$. Usa los pares observados y la posterior de cada patrón: la masa en $U$ de un patrón es $n_\gamma (1 - p(\gamma))$, cuántos de sus pares el modelo atribuye a personas distintas, y la masa en $M$ es $n_\gamma\, p(\gamma)$. Con eso, para un umbral $t$,

$$
\hat\mu(t) = \frac{\sum_{\gamma : w(\gamma) \geq t} n_\gamma \big(1 - p(\gamma)\big)}{\sum_{\gamma} n_\gamma \big(1 - p(\gamma)\big)},
\qquad
\hat\lambda(t) = \frac{\sum_{\gamma : w(\gamma) \leq t} n_\gamma\, p(\gamma)}{\sum_{\gamma} n_\gamma\, p(\gamma)} .
$$

Si el modelo es correcto, $n_\gamma / N_s \to P(\gamma)$ y $P(\gamma) (1 - p(\gamma)) = (1 - \pi) u(\gamma)$, así que $\hat\mu(t) \to \sum_{\gamma : w(\gamma) \geq t} u(\gamma) = P(w \geq t \mid U)$; lo mismo para $\hat\lambda$. Es decir, estiman las tasas de la sección 6.1, pero las dos dependen del modelo a través de $p(\gamma)$: si el modelo está mal, las tasas estimadas también.

Las tasas son sobre los pares de $\mathcal P_s$. Un par de la misma persona que el filtro dejó afuera (sección 3.4) es un vínculo perdido que $\hat\lambda$ no cuenta.

### 6.3 Umbrales discretos: nunca pasarse

Los pesos toman pocos valores, uno por patrón observado, así que en general no hay un umbral que dé exactamente $\mu$ o $\lambda$. El código no aleatoriza. Con $\mathcal W$ el conjunto de pesos distintos de los patrones observados (redondeados a 12 decimales, para que patrones con el mismo peso queden juntos),

$$
T_\mu = \min\{ t \in \mathcal W : \hat\mu(t) \leq \mu \},
\qquad
T_\lambda = \max\{ t \in \mathcal W : \hat\lambda(t) \leq \lambda \} .
$$

Como $\hat\mu$ es no creciente en $t$, el conjunto de la izquierda es un segmento final de $\mathcal W$, y el código lo encuentra recorriendo los pesos de mayor a menor y acumulando masa en $U$ hasta que la próxima suma pasaría $\mu$. Lo mismo para $T_\lambda$ desde abajo. Las tasas que quedan, `mu_estimado` $= \hat\mu(T_\mu) \leq \mu$ (salvo con cruce, sección 6.5) y `lambda_estimado` $= \hat\lambda(T_\lambda) \leq \lambda$, se guardan en `parametros.json`. Si $T_\mu = +\infty$, `mu_estimado` es 0; si $T_\lambda = -\infty$, `lambda_estimado` es 0.

Sin aleatorización, la regla es óptima para las tasas que alcanza, $\hat\mu(T_\mu)$ y $\hat\lambda(T_\lambda)$, no necesariamente para las pedidas: una regla aleatorizada podría llegar a $\mu$ y $\lambda$ y revisar un poco menos.

### 6.4 Casos $\pm\infty$

- Si ni el peso más alto cumple $\hat\mu(t) \leq \mu$, el conjunto es vacío y $T_\mu = +\infty$: no se vincula nada.
- Si ni el más bajo cumple $\hat\lambda(t) \leq \lambda$, $T_\lambda = -\infty$: no se descarta nada.
- Con los dos, todo va a revisión.

### 6.5 Cruce de umbrales

Si $T_\lambda \geq T_\mu$, las zonas se superponen: los pesos de $[T_\mu, T_\lambda]$ cumplirían a la vez la condición de vincular y la de no vincular. Es el caso en que el resultado de Fellegi y Sunter no se aplica, porque se pueden cumplir las dos tasas sin revisar nada. En este dominio un vínculo falso es el peor error, así que gana no vincular. La regla que aplica `decidir` es, en todos los casos,

$$
d(w) =
\begin{cases}
\text{no vincular} & \text{si } w \leq T_\lambda,\\
\text{vincular} & \text{si no, y } w \geq T_\mu,\\
\text{revisar} & \text{en otro caso},
\end{cases}
$$

con $w$ redondeado a 12 decimales, igual que en $\mathcal W$. Si no hay cruce, coincide con la de la sección 6.1. Si hay cruce, se vincula sólo con $w > T_\lambda$ y no hay revisión. Por eso, con cruce, `mu_estimado` es la tasa de esa regla y no $\hat\mu(T_\mu)$:

$$
\hat\mu_{\text{cruce}} = \frac{\sum_{\gamma : w(\gamma) > T_\lambda} n_\gamma \big(1 - p(\gamma)\big)}{\sum_{\gamma} n_\gamma \big(1 - p(\gamma)\big)} \ \leq\ \hat\mu(T_\mu) \ \leq\ \mu ,
$$

donde la primera desigualdad vale porque $\{w > T_\lambda\} \subseteq \{w \geq T_\mu\}$ cuando $T_\lambda \geq T_\mu$. `lambda_estimado` sigue siendo $\hat\lambda(T_\lambda)$, porque la zona de no vincular es la misma.

### 6.6 Alcance

Las decisiones son por par. No hay transitividad, un registro puede quedar vinculado con varios de otra fuente, y nada de esto se aplica al grafo. Las tasas son estimadas bajo el modelo, sin casos etiquetados que permitan calibrarlas.

In [ ]:
# Contrato: regla de decisión con un ejemplo de un campo resuelto a mano (secciones 6.2 a 6.5).
from fractions import Fraction

from zona4_graph_loader.fellegi_sunter.decision import (
    NO_VINCULAR, REVISAR, VINCULAR, Umbrales, calcular_umbrales, decidir,
)

# Un campo de tres niveles: pesos log2(8) = 3, log2(1) = 0 y log2(1/8) = −3.
P1 = Parametros(pi=0.1, m=[[0.8, 0.1, 0.1]], u=[[0.1, 0.1, 0.8]])
C1 = {(0,): 100, (1,): 100, (2,): 800}
for g, w in (((0,), 3), ((1,), 0), ((2,), -3)):
    assert math.isclose(peso(g, P1), w, abs_tol=1e-12)

# A mano, con fracciones: p = π m / (π m + (1 − π) u) = 8/17, 1/10, 1/73.
F = Fraction
pi_, m_, u_, n_ = F(1, 10), [F(8, 10), F(1, 10), F(1, 10)], [F(1, 10), F(1, 10), F(8, 10)], [100, 100, 800]
p_ = [pi_ * m_[l] / (pi_ * m_[l] + (1 - pi_) * u_[l]) for l in range(3)]
assert p_ == [F(8, 17), F(1, 10), F(1, 73)]
masa_u = [n_[l] * (1 - p_[l]) for l in range(3)]
masa_m = [n_[l] * p_[l] for l in range(3)]
mu_hat = {3: masa_u[0] / sum(masa_u), 0: (masa_u[0] + masa_u[1]) / sum(masa_u), -3: F(1)}
lam_hat = {-3: masa_m[2] / sum(masa_m), 0: (masa_m[2] + masa_m[1]) / sum(masa_m), 3: F(1)}
# μ̂(3) ≈ 0.057, μ̂(0) ≈ 0.153; λ̂(−3) ≈ 0.161, λ̂(0) ≈ 0.308.
assert 0.05 < mu_hat[3] < 0.06 < 0.15 < mu_hat[0] < 0.16
assert 0.16 < lam_hat[-3] < 0.17 < 0.30 < lam_hat[0] < 0.31

# Caso sin cruce: μ = 0.06, λ = 0.2 → T_μ = 3, T_λ = −3.
u1 = calcular_umbrales(C1, P1, mu=0.06, lam=0.2)
assert math.isclose(u1.t_mu, 3) and math.isclose(u1.t_lambda, -3)
assert math.isclose(u1.mu_estimado, float(mu_hat[3])) and math.isclose(u1.lambda_estimado, float(lam_hat[-3]))
assert [decidir(peso(g, P1), u1) for g in ((0,), (1,), (2,))] == [VINCULAR, REVISAR, NO_VINCULAR]

# Nunca pasarse: con μ apenas menor que μ̂(3) no se vincula nada (T_μ = +∞);
# con λ apenas menor que λ̂(−3) no se descarta nada (T_λ = −∞).
u2 = calcular_umbrales(C1, P1, mu=float(mu_hat[3]) - 1e-9, lam=float(lam_hat[-3]) - 1e-9)
assert u2.t_mu == math.inf and u2.t_lambda == -math.inf
assert u2.mu_estimado == 0.0 and u2.lambda_estimado == 0.0
assert all(decidir(peso(g, P1), u2) == REVISAR for g in C1), "con ±∞ todo va a revisión"

# Cruce: μ = 0.2 y λ = 0.4 dan T_μ = 0 y T_λ = 0. Gana no vincular.
u3 = calcular_umbrales(C1, P1, mu=0.2, lam=0.4)
assert math.isclose(u3.t_mu, 0, abs_tol=1e-12) and math.isclose(u3.t_lambda, 0, abs_tol=1e-12)
assert u3.t_lambda >= u3.t_mu
assert [decidir(peso(g, P1), u3) for g in ((0,), (1,), (2,))] == [VINCULAR, NO_VINCULAR, NO_VINCULAR]
# Con cruce, mu_estimado es la tasa de la regla aplicada: Σ_{w > T_λ} masa_U / total_U = μ̂(3), no μ̂(T_μ) = μ̂(0).
assert math.isclose(u3.mu_estimado, float(mu_hat[3])) and mu_hat[3] < mu_hat[0]
assert math.isclose(u3.lambda_estimado, float(lam_hat[0]))

# Cruce armado a mano: entre T_μ y T_λ, no vincular.
cruzados = Umbrales(t_mu=-1.0, t_lambda=1.0, mu_estimado=0.0, lambda_estimado=0.0)
assert decidir(0.0, cruzados) == NO_VINCULAR and decidir(1.0, cruzados) == NO_VINCULAR
assert decidir(1.5, cruzados) == VINCULAR

## 7. Resultados con los datos del repo

Esta sección lee `data/processed/fellegi_sunter/parametros.json`, la salida de una corrida real, y muestra sólo agregados por estrato: no hay nombres ni claves en ese archivo. Sirve para revisar que los parámetros estimados tengan sentido. Estas celdas no son de contrato: dependen de los datos y de cuándo se corrió el modelo.

Si el archivo no existe, generalo con el comando de la sección 0. Qué mirar:

- **Pares y $\pi$.** $\pi N_s$ es el número esperado de pares de la misma persona en el estrato. Si las dos fuentes no tienen duplicados y cada persona aparece a lo sumo una vez en cada una, no puede superar al número de registros de la fuente más chica. Si lo supera, hay duplicados en alguna fuente o el modelo está atribuyendo a $M$ pares que no son de la misma persona (por ejemplo, familiares con el mismo apellido y la misma fecha de secuestro).
- **Apellido extra.** Cuántos registros de cada fuente tienen apellido completo distinto del base (sección 2.1). Si una corrida no lo trae, es anterior a ese cambio.
- **Tablas de $m$, $u$ y $w$.** En $M$, los niveles de más acuerdo tendrían que ser más probables que en $U$ y sus pesos positivos. Un campo con $m_k = u_k = 1/L_k$ en todos los niveles no fue observado en el estrato. El apellido `distinto` sólo tiene el pseudo-conteo (sección 5.3).
- **Umbrales y decisiones.** Si $T_\lambda \geq T_\mu$ hay cruce (sección 6.5) y no queda nada para revisar.
- **Estabilidad.** Si los inicios llegan a objetivos distintos, el máximo elegido puede no ser global; si llegan al mismo objetivo con $\pi$ distintos, hay un problema de identificabilidad.

In [ ]:
import json
import math

from zona4_graph_loader.fellegi_sunter.comparaciones import CAMPOS, NIVELES
from zona4_graph_loader.fellegi_sunter.em import PSEUDO_CONTEO

SALIDA = RAIZ / "data" / "processed" / "fellegi_sunter"
RUTA_PARAMETROS = SALIDA / "parametros.json"
resumen = json.loads(RUTA_PARAMETROS.read_text(encoding="utf-8")) if RUTA_PARAMETROS.exists() else None

if resumen is None:
    print("No está", RUTA_PARAMETROS)
    print("Generalo con: PYTHONPATH=src .venv/Scripts/python -m zona4_graph_loader.fellegi_sunter.correr")
else:
    registros = resumen["registros_por_fuente"]
    print(f"μ objetivo = {resumen['mu']}, λ objetivo = {resumen['lambda']}\n")
    print("Registros por fuente:")
    for fuente, n in sorted(registros.items()):
        print(f"  {fuente:20s} {n:6d}")
    extra = resumen.get("registros_con_apellido_extra")
    if extra is None:
        print("\nEsta corrida no trae registros_con_apellido_extra (es anterior al apellido completo del Parque).")
    else:
        print("\nRegistros con apellido extra (sección 2.1):")
        for fuente, n in sorted(registros.items()):
            print(f"  {fuente:20s} {extra.get(fuente, 0):6d} de {n:6d}")
    print()
    print(f"{'estrato':45s} {'pares':>7s} {'patrones':>8s} {'π':>8s} {'πN':>8s} {'cota 1 a 1':>10s} {'iter':>5s} {'convergió':>9s}")
    for clave, e in resumen["estratos"].items():
        p = e["parametros"]
        esperados = p["pi"] * e["pares"]
        cota = min(registros.get(f, 0) for f in clave.split("|"))
        marca = "  <- πN supera la cota" if esperados > cota else ""
        print(f"{clave:45s} {e['pares']:7d} {e['patrones_distintos']:8d} {p['pi']:8.4f} {esperados:8.1f} "
              f"{cota:10d} {p['iteraciones']:5d} {str(p['convergio']):>9s}{marca}")

In [ ]:
# m, u y peso por nivel, por estrato y campo.
if resumen is not None:
    for clave, e in resumen["estratos"].items():
        p = e["parametros"]
        print(f"=== {clave}")
        for campo in CAMPOS:
            m, u = p["m"][campo], p["u"][campo]
            L_k = len(NIVELES[campo])
            if all(math.isclose(m[n], 1 / L_k) and math.isclose(u[n], 1 / L_k) for n in NIVELES[campo]):
                print(f"  {campo}: nunca observado en este estrato (m = u = 1/{L_k})")
                continue
            print(f"  {campo}:")
            for nivel in NIVELES[campo]:
                nota = "   (no aparece: filtro del apellido)" if (campo, nivel) == ("apellido", "distinto") else ""
                print(f"    {nivel:20s} m = {m[nivel]:.4f}   u = {u[nivel]:.4f}   w = {math.log2(m[nivel] / u[nivel]):+7.2f} bits{nota}")
        print()

In [ ]:
# Controles que se pueden hacer sólo con parametros.json.
if resumen is not None:
    alfa = PSEUDO_CONTEO
    L_ap = len(NIVELES["apellido"])
    print(f"{'estrato':45s} {'campos observados':>18s}  controles")
    for clave, e in resumen["estratos"].items():
        p, N = e["parametros"], e["pares"]
        # Sección 5.3: el apellido 'distinto' sólo tiene el pseudo-conteo.
        m4 = alfa / (L_ap * alfa + p["pi"] * N)
        u4 = alfa / (L_ap * alfa + (1 - p["pi"]) * N)
        ok_m4 = math.isclose(p["m"]["apellido"]["distinto"], m4, rel_tol=1e-6)
        ok_u4 = math.isclose(p["u"]["apellido"]["distinto"], u4, rel_tol=1e-6)
        # Sección 5.8: campos observados (no clavados en 1/L_k).
        observados = [
            c for c in CAMPOS
            if not all(math.isclose(p["m"][c][n], 1 / len(NIVELES[c])) and math.isclose(p["u"][c][n], 1 / len(NIVELES[c]))
                       for n in NIVELES[c])
        ]
        # Etiquetas (sección 5.7): Δ = Σ_k (m_k(0) − u_k(0)) ≥ 0.
        delta = sum(p["m"][c][NIVELES[c][0]] - p["u"][c][NIVELES[c][0]] for c in CAMPOS)
        avisos = []
        if not (ok_m4 and ok_u4):
            avisos.append("m o u del apellido 'distinto' no es sólo el pseudo-conteo")
        if len(observados) < 3:
            avisos.append("menos de 3 campos observados: no identificable")
        elif len(observados) == 3:
            avisos.append("exactamente 3 campos: en el límite de identificabilidad")
        if delta < 0:
            avisos.append("Δ < 0: etiquetas sin ordenar")
        print(f"{clave:45s} {len(observados):18d}  Δ = {delta:+.3f}; " + ("; ".join(avisos) or "sin avisos"))

In [ ]:
# Umbrales y decisiones por estrato.
if resumen is not None:
    print(f"{'estrato':45s} {'T_μ':>8s} {'T_λ':>8s} {'μ̂':>7s} {'λ̂':>7s}  {'vincular':>9s} {'revisar':>9s} {'no vinc.':>9s}")
    for clave, e in resumen["estratos"].items():
        um, d = e["umbrales"], e["decisiones"]
        assert sum(d.values()) == e["pares"], f"{clave}: las decisiones no suman los pares"
        t_mu, t_lam = float(um["t_mu"]), float(um["t_lambda"])
        cruce = "  <- cruce: gana no vincular (6.5)" if t_lam >= t_mu else ""
        print(f"{clave:45s} {t_mu:8.2f} {t_lam:8.2f} {um['mu_estimado']:7.4f} {um['lambda_estimado']:7.4f}  "
              f"{d['vincular']:9d} {d['revisar']:9d} {d['no_vincular']:9d}{cruce}")

In [ ]:
# Estabilidad entre inicios del EM (sección 5.6).
if resumen is not None:
    print(f"{'estrato':45s} {'inicios':>7s} {'max |J − J*|':>13s} {'rango de π':>12s} {'iguales a J*':>13s}")
    for clave, e in resumen["estratos"].items():
        J, pis = e["objetivo_por_inicio"], e["pi_por_inicio"]
        J_mejor = max(J)
        iguales = sum(1 for j in J if abs(j - J_mejor) < 1e-6 * max(1.0, abs(J_mejor)))
        print(f"{clave:45s} {len(J):7d} {max(J_mejor - j for j in J):13.3g} {max(pis) - min(pis):12.3g} {iguales:13d}")

**Lo que mostraba la corrida guardada en el repo al escribir este notebook (noviembre de 2026).** Puede cambiar si se vuelve a correr; las celdas de arriba siempre muestran el archivo actual.

- Había cuatro estratos: `archivo_memoria` con `detalles_personas` y con `minjus_victimas`, `detalles_personas` con `minjus_victimas`, y `juicios_condenados` con `minjus_imputados`. Los otros seis pares de fuentes cruzan una fuente de víctimas con una de imputados o condenados y no dejan pares, por la condición de roles.
- Todos los inicios del EM llegaron al mismo objetivo y al mismo $\pi$ en los cuatro estratos.
- En `archivo_memoria` con `detalles_personas`, $\pi N_s$ y los pares vinculados superaban la cantidad de registros de `archivo_memoria`. Hay que mirarlo antes de confiar en ese estrato: o hay duplicados en `detalles_personas`, o el modelo está metiendo en $M$ pares que no son de la misma persona.
- En `juicios_condenados` con `minjus_imputados` los umbrales se cruzaban y no quedaba nada para revisar. Dos estratos tenían exactamente tres campos observados.
- En `detalles_personas` con `minjus_victimas` el centro estaba observado en una fracción muy chica de los pares, así que su $m$ y su $u$ se apoyan en pocos pares. En todos los estratos había registros vinculados con más de un registro de la otra fuente (sección 8).

## 8. Cómo auditar un par de `pares.csv`

Esta sección explica cómo reconstruir, para cualquier fila de `pares.csv`, el peso, la posterior y la decisión a partir de `parametros.json` y de las fórmulas de las secciones 4 y 6. Es la forma de verificar que un par concreto se decidió por las razones que dice el modelo. Las celdas no muestran filas reales.

Columnas de `pares.csv`, una fila por par de $\mathcal P$:

| columna | contenido |
|---|---|
| `clave_a`, `clave_b` | `persona_key` de los dos registros |
| `fuente_a`, `fuente_b` | sus fuentes; el estrato es el par ordenado alfabéticamente, unido con una barra vertical |
| `nombre_a`, `nombre_b` | el nombre tal como viene en la fuente (por esto el archivo queda fuera de git) |
| `nivel_<campo>` | nombre del nivel $\gamma_k$ de cada campo, o `falta` si $\gamma_k = \varnothing$ |
| `peso_<campo>` | $w_k(\gamma_k)$ redondeado a 4 decimales, vacío si falta |
| `peso` | $w(\gamma)$ redondeado a 4 decimales |
| `prob_match` | $p(\gamma)$ redondeada a 6 decimales |
| `decision` | `vincular`, `revisar` o `no_vincular` |

Receta para un par:

1. Buscar el estrato en `parametros.json` y, para cada campo observado, leer $m_k(\ell)$ y $u_k(\ell)$ del nivel. Recalcular $w_k = \log_2(m_k / u_k)$ y comparar con `peso_<campo>`.
2. Sumar los $w_k$ de los campos observados y comparar con `peso` (sección 4.4).
3. Recalcular $p = 1 / (1 + e^{-(\operatorname{logit} \pi + w \ln 2)})$ y comparar con `prob_match`.
4. Aplicar la regla de la sección 6.5 con $T_\mu$ y $T_\lambda$ del estrato y comparar con `decision`.
5. Mirar qué campos faltan: cada uno aporta 0 bits. Un par con pocos campos observados puede quedar en revisión aunque todo lo observado coincida.

La celda siguiente implementa la receta con las fórmulas del texto, sin usar las funciones del modelo, y la prueba con un par inventado.

In [ ]:
def reconstruir(fila, resumen):
    # Peso por campo, peso total, posterior y decisión de una fila de pares.csv, desde parametros.json.
    clave = "|".join(sorted((fila["fuente_a"], fila["fuente_b"])))
    e = resumen["estratos"][clave]
    p = e["parametros"]
    pesos = {}
    for campo in CAMPOS:
        nivel = fila[f"nivel_{campo}"]
        pesos[campo] = None if nivel == "falta" else math.log2(p["m"][campo][nivel] / p["u"][campo][nivel])
    w = sum(x for x in pesos.values() if x is not None)
    logit_p = math.log(p["pi"] / (1 - p["pi"])) + w * math.log(2)
    prob = 1 / (1 + math.exp(-logit_p))
    t_mu, t_lam = float(e["umbrales"]["t_mu"]), float(e["umbrales"]["t_lambda"])
    w_r = round(w, 12)
    decision = "no_vincular" if w_r <= t_lam else ("vincular" if w_r >= t_mu else "revisar")
    return {"estrato": clave, "pesos": pesos, "peso": w, "prob_match": prob, "decision": decision}


if resumen is not None:
    # Un par inventado (no es una fila real) en el primer estrato.
    clave_ejemplo = next(iter(resumen["estratos"]))
    fuente_a, fuente_b = clave_ejemplo.split("|")
    inventada = {
        "fuente_a": fuente_a, "fuente_b": fuente_b,
        "nivel_apellido": "exacto", "nivel_nombre": "exacto", "nivel_fecha_nacimiento": "falta",
        "nivel_fecha_secuestro": "mismo_año", "nivel_centro": "falta",
    }
    r = reconstruir(inventada, resumen)
    print("estrato:", r["estrato"])
    for campo, x in r["pesos"].items():
        print(f"  {campo:18s} {inventada['nivel_' + campo]:20s} " + ("falta (0 bits)" if x is None else f"{x:+.3f} bits"))
    print(f"  peso = {r['peso']:+.3f} bits, P(M|γ) = {r['prob_match']:.4f}, decisión = {r['decision']}")

La celda siguiente aplica `reconstruir` a todas las filas de `pares.csv`, si existe, y muestra sólo agregados por estrato: cuántas filas no coinciden con la reconstrucción, la fracción de pares con cada campo observado, la tasa efectiva de vínculos falsos y de vínculos perdidos de las decisiones tomadas (con las masas de la sección 6.2; tienen que coincidir con `mu_estimado` y `lambda_estimado` de `parametros.json`, también con cruce) y cuántos registros quedaron vinculados con más de un registro de la otra fuente. Para la comparación se toleran los redondeos del archivo.

In [ ]:
import csv
from collections import Counter, defaultdict

RUTA_PARES = SALIDA / "pares.csv"
if resumen is None or not RUTA_PARES.exists():
    print("No está", RUTA_PARES, "o falta parametros.json; generalos con el comando de la sección 0.")
else:
    filas = Counter()
    distintas = Counter()
    observados = defaultdict(Counter)
    masa = defaultdict(lambda: {"u_total": 0.0, "m_total": 0.0, "u_vinc": 0.0, "m_no_vinc": 0.0})
    vinculos = defaultdict(Counter)
    with RUTA_PARES.open(encoding="utf-8", newline="") as f:
        for fila in csv.DictReader(f):
            r = reconstruir(fila, resumen)
            s = r["estrato"]
            filas[s] += 1
            ok = (
                all((fila[f"peso_{c}"] == "") == (r["pesos"][c] is None) for c in CAMPOS)
                and all(r["pesos"][c] is None or abs(float(fila[f"peso_{c}"]) - r["pesos"][c]) <= 6e-5 for c in CAMPOS)
                and abs(float(fila["peso"]) - r["peso"]) <= 6e-5
                and abs(float(fila["prob_match"]) - r["prob_match"]) <= 6e-7
                and fila["decision"] == r["decision"]
            )
            distintas[s] += not ok
            for c in CAMPOS:
                observados[s][c] += fila[f"nivel_{c}"] != "falta"
            p = r["prob_match"]
            masa[s]["u_total"] += 1 - p
            masa[s]["m_total"] += p
            if r["decision"] == "vincular":
                masa[s]["u_vinc"] += 1 - p
                vinculos[s][("a", fila["clave_a"])] += 1
                vinculos[s][("b", fila["clave_b"])] += 1
            elif r["decision"] == "no_vincular":
                masa[s]["m_no_vinc"] += p

    print(f"{'estrato':45s} {'filas':>7s} {'no coinciden':>12s} {'μ̂ efectiva':>11s} {'μ̂ guardada':>11s} {'λ̂ efectiva':>11s} {'λ̂ guardada':>11s} {'registros con >1 vínculo':>25s}")
    for s in sorted(filas):
        mu_ef = masa[s]["u_vinc"] / masa[s]["u_total"]
        lam_ef = masa[s]["m_no_vinc"] / masa[s]["m_total"]
        multiples = sum(1 for n in vinculos[s].values() if n > 1)
        um = resumen["estratos"][s]["umbrales"]
        print(f"{s:45s} {filas[s]:7d} {distintas[s]:12d} {mu_ef:11.4f} {um['mu_estimado']:11.4f} "
              f"{lam_ef:11.4f} {um['lambda_estimado']:11.4f} {multiples:25d}")
    print()
    print("Fracción de pares con cada campo observado:")
    print(f"{'estrato':45s} " + " ".join(f"{c[:14]:>14s}" for c in CAMPOS))
    for s in sorted(filas):
        print(f"{s:45s} " + " ".join(f"{observados[s][c] / filas[s]:14.3f}" for c in CAMPOS))

## 9. Límites, supuestos que pueden fallar y próximos pasos

Esta sección junta lo que el modelo supone y dónde puede fallar. Todo resultado de las secciones 7 y 8 hay que leerlo con esta lista al lado.

**Independencia condicional (4.1).** Dada la clase, los campos se suponen independientes. Hay razones para dudar: entre personas distintas, compartir apellido aumenta la chance de compartir fecha de secuestro y centro (familias secuestradas juntas), y entre registros de la misma persona, una fuente que copia de otra hace coincidir varios campos a la vez. Si la dependencia es positiva dentro de $U$, el peso suma evidencia que no es independiente y sobreestima la razón de verosimilitud: produce vínculos falsos con más confianza de la debida.

**Faltantes ignorables (4.2).** Se supone que qué campos faltan no depende de la clase ni de los valores faltantes. Lo estructural (qué registra cada fuente) es constante dentro de un estrato y no molesta. Lo que puede fallar es la variación dentro de una fuente: si las personas con fechas registradas son también las que aparecen en más fuentes, la presencia de la fecha informa sobre la clase y el modelo la ignora.

**Pares independientes (4.3).** La verosimilitud trata a los pares como independientes y no impone que cada registro se empareje con a lo sumo uno de la otra fuente. El modelo puede vincular un registro con varios.

**El filtro (3.4).** Los pares con apellido base `distinto` no se comparan y cuentan como personas distintas; también los que sólo comparten un apellido que el Parque trae en el nombre largo. Eso incluye cambios de apellido, transposiciones, dos letras de diferencia y erratas en apellidos de menos de $\tau$ letras. Las tasas $\hat\lambda$ no cuentan esos vínculos perdidos.

**Apellido completo del Parque (2.1).** El nombre largo no separa nombres de apellidos, y la regla "todo token nuevo es apellido" tiene costos:

- Con dos o tres tokens extra suele tratarse del apellido materno más el de casada ("Mrad Elias Medina"). Si la otra fuente registra el de casada y el Parque no, o al revés, el nivel es `contenido` y no `exacto`.
- La misma persona registrada en otra fuente con un solo apellido ahora da `contenido` contra la ficha del Parque, donde antes daba `exacto`. Como $m$ y $u$ se estiman por estrato, el modelo aprende cuánto vale `contenido` en cada uno, pero el nivel mezcla "el otro registro omitió un apellido" con "tiene un apellido distinto de más".
- Un token del nombre de pila escrito distinto en el largo que en el corto se suma como apellido, y puede bajar el nivel del apellido.

**Niveles (2).** Los niveles de fecha mezclan acuerdo parcial con precisión parcial. Una fecha imposible se trata como faltante, igual que un centinela. Un valor no vacío en el nodo que no es fecha tapa una fecha válida de la arista (3.1); eso está en `identity_resolution._fechas_de`, que es código del loader y este módulo reusa tal cual.

**Identificabilidad y estabilidad (5.8).** Los estratos con tres campos observados están en el límite. Que todos los inicios lleguen al mismo objetivo es una señal favorable, no una prueba.

**Decisiones por par (6.6).** No hay transitividad ni restricción uno a uno, y las decisiones no se aplican al grafo.

**Sin etiquetas no hay calibración.** $p(\gamma)$, $\hat\mu$ y $\hat\lambda$ son las que da el modelo; no hay casos con la verdad conocida para comparar. Revisar a mano una muestra de pares, estratificada por decisión y por peso, daría una primera medida de calibración y de las tasas reales.

**Próximos pasos.**

- Revisar a mano una muestra de pares de cada estrato y decisión, para estimar las tasas reales de error y ver si la posterior está calibrada.
- Mirar los estratos donde $\pi N_s$ supera la cota uno a uno.
- Evaluar modelos que imponen coherencia entre pares: emparejamientos uno a uno entre dos fuentes ([Sadinle (2017)](https://doi.org/10.1080/01621459.2016.1148612)), vinculación entre más de dos fuentes con transitividad ([Sadinle y Fienberg (2013)](https://doi.org/10.1080/01621459.2012.757231)) y particiones bayesianas de registros de varias fuentes con duplicados ([Aleshin-Guendel y Sadinle (2023)](https://doi.org/10.1080/01621459.2021.2013242)), implementadas en el paquete de R [multilink](https://github.com/aleshing/multilink). [Sadinle (2014)](https://doi.org/10.1214/14-AOAS779) trabaja con un registro de homicidios y [Sadinle (2018)](https://doi.org/10.1214/18-AOAS1178) con registros de violaciones de derechos humanos, problemas cercanos a este.

## Bibliografía

Cada obra con su DOI y, cuando hay, la versión de arXiv. Todos los links se verificaron al escribir este notebook (noviembre de 2026).

- Fellegi, I. P. y Sunter, A. B. (1969). A theory for record linkage. *Journal of the American Statistical Association*, 64(328), 1183–1210. https://doi.org/10.1080/01621459.1969.10501049
- Dempster, A. P., Laird, N. M. y Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society, Series B (Methodological)*, 39(1), 1–22. https://doi.org/10.1111/j.2517-6161.1977.tb01600.x
- Winkler, W. E. (1988). Using the EM algorithm for weight computation in the Fellegi–Sunter model of record linkage. *Proceedings of the Section on Survey Research Methods, American Statistical Association*, 667–671. Sin DOI. Versión revisada: U.S. Census Bureau, Statistical Research Report RR2000/05 (2000), https://www.census.gov/library/working-papers/2000/adrm/rr2000-05.html
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. https://doi.org/10.1093/biomet/63.3.581
- Allman, E. S., Matias, C. y Rhodes, J. A. (2009). Identifiability of parameters in latent structure models with many observed variables. *The Annals of Statistics*, 37(6A), 3099–3132. https://doi.org/10.1214/09-AOS689 ; arXiv: https://arxiv.org/abs/0809.5032
- Sadinle, M. y Fienberg, S. E. (2013). A generalized Fellegi–Sunter framework for multiple record linkage with application to homicide record systems. *Journal of the American Statistical Association*, 108(502), 385–397. https://doi.org/10.1080/01621459.2012.757231 ; arXiv: https://arxiv.org/abs/1205.3217
- Sadinle, M. (2014). Detecting duplicates in a homicide registry using a Bayesian partitioning approach. *The Annals of Applied Statistics*, 8(4), 2404–2434. https://doi.org/10.1214/14-AOAS779 ; arXiv: https://arxiv.org/abs/1407.8219
- Sadinle, M. (2017). Bayesian estimation of bipartite matchings for record linkage. *Journal of the American Statistical Association*, 112(518), 600–612. https://doi.org/10.1080/01621459.2016.1148612 ; arXiv: https://arxiv.org/abs/1601.06630
- Sadinle, M. (2018). Bayesian propagation of record linkage uncertainty into population size estimation of human rights violations. *The Annals of Applied Statistics*, 12(2), 1013–1038. https://doi.org/10.1214/18-AOAS1178 ; arXiv: https://arxiv.org/abs/1812.09590
- Aleshin-Guendel, S. y Sadinle, M. (2023). Multifile partitioning for record linkage and duplicate detection. *Journal of the American Statistical Association*, 118(543), 1786–1795 (publicado en línea en 2022). https://doi.org/10.1080/01621459.2021.2013242 ; arXiv: https://arxiv.org/abs/2110.03839 ; paquete de R multilink: https://github.com/aleshing/multilink